# Horizon Guard v6 — версия для Google Colab

Ноутбук обучает ансамбль V4, residual/source-коррекцию V5 и Horizon Guard V6, затем сохраняет `submission.csv` в `/content`.

Нужны только три файла: `train.csv`, `test.csv`, `sample_submission.csv`.

**Версия для сдачи с выборочными пояснениями.** В начале каждого кодового модуля описана его роль; перед каждой функцией и классом приведены назначение, входы и результат. Повторяющиеся построчные комментарии к спискам, параметрам и однотипным операциям удалены. Вычислительный путь сохранён; подробности находятся в `Аудит_упрощения_horizon_guard_colab_clean.md`.


## 1. Установка и импорты

Загрузите CSV в `/content`, затем выполните **Runtime → Run all**


In [2]:
# МОДУЛЬ 1. Фиксация окружения Colab.
# Устанавливает точные версии библиотек, чтобы обучение и итоговый CSV воспроизводились в одной среде.

!pip install --upgrade numpy==2.5.0 pandas==3.0.4 scipy==1.18.0 lightgbm==4.6.0 python-dateutil==2.9.0.post0 pytz==2026.2 tzdata==2026.2 six==1.17.0


In [3]:
# МОДУЛЬ 1. Импорты и общие зависимости.
# Подключает стандартные библиотеки, NumPy, pandas и LightGBM для всех следующих этапов pipeline.

from __future__ import annotations

import gc
import math
import re
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Mapping, Sequence

import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display
from pandas.api.types import is_numeric_dtype

# Colab хранит загруженные файлы в /content.
DATA_DIR = Path("/content")
OUTPUT_DIR = DATA_DIR
SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"
NUM_THREADS = 4

REQUIRED_FILES = ("train.csv", "test.csv", "sample_submission.csv")
missing_files = [name for name in REQUIRED_FILES if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        "Загрузите в /content следующие файлы: " + ", ".join(missing_files)
    )

print(f"LightGBM {lgb.__version__}; pandas {pd.__version__}; NumPy {np.__version__}")


LightGBM 4.6.0; pandas 3.0.4; NumPy 2.5.0


## 2. Чтение и подготовка данных


In [4]:
# МОДУЛЬ 2. Контракты данных, метрика и строгий CSV-парсер.
# Определяет служебные поля и проверки, которые защищают схему, порядок id и числовые значения.

ID_COLUMN = "id"
SNAPSHOT_COLUMN = "dt"
TARGET_COLUMN = "target"
WEIGHT_COLUMN = "w"
FORBIDDEN_FEATURE_COLUMNS = frozenset(
    {ID_COLUMN, SNAPSHOT_COLUMN, TARGET_COLUMN, WEIGHT_COLUMN}
)
CATEGORY_COLUMNS = ("gender", "adminarea", "city_smart_name", "addrref")
HIGH_CARD_TEXT_COLUMNS = ("dp_ewb_last_employment_position",)
STRUCTURED_COLUMNS = ("dp_address_unique_regions",)
DATE_COLUMNS = ("period_last_act_ad",)
SPECIAL_FEATURE_COLUMNS = (
    *CATEGORY_COLUMNS,
    *HIGH_CARD_TEXT_COLUMNS,
    *STRUCTURED_COLUMNS,
    *DATE_COLUMNS,
)
INCOME_PROXY_COLUMNS = (
    "incomeValue",
    "salary_6to12m_avg",
    "dp_ils_avg_salary_1y",
    "dp_ils_avg_salary_2y",
    "dp_ils_avg_salary_3y",
    "dp_payoutincomedata_payout_avg_3_month",
    "dp_payoutincomedata_payout_avg_6_month",
    "dp_payoutincomedata_payout_avg_prev_year",
    "salary_median_in_gex_r1",
    "profit_income_out_rur_amt_9m",
    "profit_income_out_rur_amt_l2m",
)
HIGH_SIGNAL_MISSING_COLUMNS = (
    "salary_6to12m_avg",
    "dp_ils_avg_salary_1y",
    "dp_ils_avg_salary_2y",
    "dp_payoutincomedata_payout_avg_3_month",
    "dp_payoutincomedata_payout_avg_6_month",
    "salary_median_in_gex_r1",
    "avg_6m_travel",
    "avg_6m_hotels",
    "avg_by_category__amount__sum__cashflowcategory_name__supermarkety",
    "hdb_outstand_sum",
)
SOURCE_FAMILY_PREFIXES: Mapping[str, tuple[str, ...]] = {
    "income": (
        "salary_",
        "dp_ils_",
        "dp_payoutincomedata_",
        "profit_income_",
        "incomeValue",
        "per_capita_income_",
    ),
    "credit_bki": ("hdb_", "bki_", "loan", "pil", "ovrd", "cred_"),
    "turnover": (
        "turn_",
        "avg_cur_",
        "avg_credit_",
        "avg_debet_",
        "by_category_",
        "transaction_category_",
        "amount_by_category_",
    ),
    "balances": (
        "curr_",
        "dda_",
        "loanacc_",
        "express_",
        "min_balance_",
        "max_balance_",
        "avg_balance_",
        "curbal_",
    ),
    "digital": ("vert_", "mob_", "device_", "sms", "cnt", "called", "businessTel"),
}
RAW_NA_VALUES = ("", "nan", "NaN", "NAN", "None", "none", "NONE", "null", "NULL", "<NA>")
MISSING_TOKEN = "__missing__"
RARE_TOKEN = "__rare__"
UNKNOWN_TOKEN = "__unknown__"


# КЛАСС `ParsedRawData`
# Контейнер результата строгого парсинга. Хранит согласованные train/test-признаки, target, веса, ID, даты,
# месяцы и список исключённых полей, чтобы следующие этапы не зависели от скрытого состояния ячеек.
# Входы: поля экземпляра и методы, объявленные ниже.
# Результат: экземпляр класса, используемый как явный контракт состояния.
@dataclass
class ParsedRawData:
    """Подготовленные train/test данные без служебных столбцов в признаках."""

    train_features: pd.DataFrame
    test_features: pd.DataFrame
    y: np.ndarray
    w: np.ndarray
    train_ids: pd.Series
    test_ids: pd.Series
    train_dates: pd.Series
    test_dates: pd.Series
    train_month: np.ndarray
    test_month: np.ndarray
    dropped_columns: tuple[str, ...]

    # ФУНКЦИЯ `ParsedRawData.feature_columns`
    # Возвращает неизменяемый кортеж имён признаков в фактическом порядке train-матрицы. Этот порядок является
    # частью контракта LightGBM и защищает от незаметной перестановки столбцов.
    # Входы: нет явных аргументов.
    # Результат: значение типа `tuple[str, ...]`.
    @property
    def feature_columns(self) -> tuple[str, ...]:
        return tuple(self.train_features.columns)


# КЛАСС `FeatureConfig`
# Неизменяемая конфигурация блоков F0–F4. Определяет, какие target-free преобразования строятся, как
# обрабатываются редкие категории и какие raw-поля исключаются.
# Входы: поля экземпляра и методы, объявленные ниже.
# Результат: экземпляр класса, используемый как явный контракт состояния.
@dataclass(frozen=True)
class FeatureConfig:
    """Настройки блоков признаков, которые обучаются только на текущем фолде."""

    include_f0_raw: bool = True
    include_f1_categories: bool = True
    include_f2_time: bool = True
    include_snapshot_month: bool = False
    include_f3_missingness: bool = True
    include_f4_financial: bool = True
    rare_min_count: int = 20
    rare_min_fraction: float = 0.0005
    one_hot_columns: tuple[str, ...] = ("gender",)
    high_signal_missing_columns: tuple[str, ...] = HIGH_SIGNAL_MISSING_COLUMNS
    excluded_raw_columns: tuple[str, ...] = ()

    # ФУНКЦИЯ `FeatureConfig.__post_init__`
    # Проверяет конфигурацию сразу после создания: допустимость порогов, отсутствие повторов и невозможность
    # маскировать служебные поля как обычные исключения. Ошибка останавливает pipeline до построения
    # признаков.
    # Входы: нет явных аргументов.
    # Результат: значение типа `None`.
    def __post_init__(self) -> None:
        if self.rare_min_count < 1:
            raise ValueError("rare_min_count must be at least 1")
        if not 0.0 <= self.rare_min_fraction <= 1.0:
            raise ValueError("rare_min_fraction must be in [0, 1]")
        if len(set(self.excluded_raw_columns)) != len(self.excluded_raw_columns):
            raise ValueError("excluded_raw_columns contains duplicates")
        leaked = FORBIDDEN_FEATURE_COLUMNS.intersection(self.excluded_raw_columns)
        if leaked:
            raise ValueError(f"cannot exclude service columns from F0: {sorted(leaked)}")


# ФУНКЦИЯ `_validate_metric_inputs`
# Приводит target, прогноз и веса к одномерным float64-векторам и проверяет длину, конечность и корректность
# весов. Возвращает три безопасных массива для единой реализации WMAE.
# Входы: `y_true`, `y_pred`, `weights`.
# Результат: значение типа `tuple[np.ndarray, np.ndarray, np.ndarray]`.
def _validate_metric_inputs(
    y_true, y_pred, weights
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    y = np.asarray(y_true, dtype=np.float64)
    pred = np.asarray(y_pred, dtype=np.float64)
    w = np.asarray(weights, dtype=np.float64)
    if y.ndim != 1 or pred.ndim != 1 or w.ndim != 1:
        raise ValueError("y_true, y_pred and weights must be one-dimensional")
    if not len(y) == len(pred) == len(w) or not len(y):
        raise ValueError("metric inputs must have the same non-zero length")
    if (
        not np.isfinite(y).all()
        or not np.isfinite(pred).all()
        or (not np.isfinite(w).all())
    ):
        raise ValueError("metric inputs must be finite")
    if (w < 0).any() or w.sum() <= 0:
        raise ValueError("weights must be non-negative with a positive sum")
    return (y, pred, w)


# ФУНКЦИЯ `wmae_official`
# Считает официальную Weighted Mean Absolute Error: сумму w·|y−prediction|, делённую на сумму весов. Чем
# меньше результат, тем лучше модель.
# Входы: `y_true`, `y_pred`, `weights`.
# Результат: значение типа `float`.
def wmae_official(y_true, y_pred, weights) -> float:
    """Официальная метрика: сумма взвешенных абсолютных ошибок, делённая на сумму весов."""
    y, pred, w = _validate_metric_inputs(y_true, y_pred, weights)
    return float(np.sum(w * np.abs(y - pred), dtype=np.float64) / w.sum(dtype=np.float64))


# ФУНКЦИЯ `month_key`
# Преобразует даты снимков в целый код year·12+month. Код сохраняет календарный порядок и позволяет вычислять
# горизонт обычной разностью.
# Входы: `dates`.
# Результат: значение типа `np.ndarray`.
def month_key(dates: pd.Series | Sequence) -> np.ndarray:
    parsed = pd.to_datetime(pd.Series(dates), errors="coerce")
    if parsed.isna().any():
        raise ValueError("snapshot dates contain missing or invalid values")
    return (parsed.dt.year * 12 + parsed.dt.month).to_numpy(dtype=np.int32)


# ФУНКЦИЯ `month_label`
# Преобразует внутренний целочисленный код месяца обратно в читаемый формат YYYY-MM. Используется в OOT-
# таблицах и диагностическом выводе.
# Входы: `key`.
# Результат: значение типа `str`.
def month_label(key: int) -> str:
    year, zero_based_month = divmod(int(key) - 1, 12)
    return f"{year:04d}-{zero_based_month + 1:02d}"


# ФУНКЦИЯ `recency_weights`
# Применяет месячное затухание к исходным весам и затем восстанавливает их прежнюю сумму. В frozen-
# конфигурации decay=1, поэтому веса численно не меняются.
# Входы: `base_weights`, `months`, `decay`.
# Результат: значение типа `np.ndarray`.
def recency_weights(base_weights, months, decay: float) -> np.ndarray:
    """Добавляет затухание по месяцам и сохраняет исходную сумму весов."""
    if not 0 < decay <= 1:
        raise ValueError(f"decay must be in (0, 1], got {decay}")
    base = np.asarray(base_weights, dtype=np.float64)
    month_array = np.asarray(months, dtype=np.int64)
    if base.ndim != 1 or month_array.ndim != 1 or len(base) != len(month_array):
        raise ValueError(
            "base_weights and months must be same-length one-dimensional arrays"
        )
    if not np.isfinite(base).all() or (base < 0).any() or base.sum() <= 0:
        raise ValueError("base weights must be finite, non-negative and have positive sum")
    age = month_array.max() - month_array
    decayed = base * np.power(float(decay), age)
    return decayed * (base.sum(dtype=np.float64) / decayed.sum(dtype=np.float64))


# ФУНКЦИЯ `_clean_string_series`
# Нормализует строковое представление пропусков: приводит к pandas string, обрезает пробелы и превращает
# пустые/служебные NA-токены в единый missing.
# Входы: `series`.
# Результат: значение типа `pd.Series`.
def _clean_string_series(series: pd.Series) -> pd.Series:
    cleaned = series.astype("string").str.strip()
    lowered = cleaned.str.casefold()
    missing_words = frozenset((value.casefold() for value in RAW_NA_VALUES if value))
    missing = lowered.isin(missing_words) | cleaned.eq("")
    return cleaned.mask(missing.fillna(False))


# ФУНКЦИЯ `normalize_text`
# Приводит текст к Unicode NFKC, единому регистру и одному пробелу между словами. Одинаковые категории
# получают одинаковый ключ на fit и transform.
# Входы: `series`.
# Результат: значение типа `pd.Series`.
def normalize_text(series: pd.Series) -> pd.Series:
    """Нормализует регистр, Unicode и пробелы одинаково на fit и transform."""
    cleaned = _clean_string_series(series)
    return (
        cleaned.str.normalize("NFKC")
        .str.casefold()
        .str.replace("\\s+", " ", regex=True)
        .str.strip()
        .mask(lambda values: values.eq(""))
    )


# ФУНКЦИЯ `_numeric_parse`
# Строго разбирает числовой столбец, поддерживая десятичную запятую. Одновременно формирует audit-record с
# долей разбора, infinity и примерами ошибочных токенов.
# Входы: `series`, `dataset`, `column`, `kind`.
# Результат: значение типа `tuple[pd.Series, dict]`.
def _numeric_parse(
    series: pd.Series, *, dataset: str, column: str, kind: str
) -> tuple[pd.Series, dict]:
    cleaned = _clean_string_series(series)
    normalized = cleaned.str.replace(",", ".", regex=False)
    parsed_nullable = pd.to_numeric(normalized, errors="coerce")
    parsed = pd.Series(
        parsed_nullable.to_numpy(dtype=np.float64, na_value=np.nan),
        index=series.index,
        name=column,
    )
    input_non_null = cleaned.notna()
    failed = input_non_null & parsed.isna()
    infinity = np.isinf(parsed.to_numpy(dtype=np.float64))
    unexpected = sorted(cleaned.loc[failed].dropna().astype(str).unique().tolist())[:10]
    non_null_count = int(input_non_null.sum())
    parsed_count = int((input_non_null & parsed.notna()).sum())
    record = {
        "dataset": dataset,
        "column": column,
        "kind": kind,
        "rows": int(len(series)),
        "input_non_null": non_null_count,
        "parsed_non_null": parsed_count,
        "failed_count": int(failed.sum()),
        "parse_rate_non_null": (
            1.0 if non_null_count == 0 else parsed_count / non_null_count
        ),
        "infinity_count": int(infinity.sum()),
        "unexpected_tokens": unexpected,
    }
    return (parsed, record)


# ФУНКЦИЯ `_parse_snapshot_dates`
# Разбирает дату снимка только в формате YYYY-MM-DD и запрещает missing/invalid значения. Временной порядок
# критичен для честного replay.
# Входы: `series`, `dataset`.
# Результат: значение типа `pd.Series`.
def _parse_snapshot_dates(series: pd.Series, *, dataset: str) -> pd.Series:
    cleaned = _clean_string_series(series)
    parsed = pd.to_datetime(cleaned, format="%Y-%m-%d", errors="coerce")
    failed = cleaned.notna() & parsed.isna()
    if failed.any() or parsed.isna().any():
        examples = sorted(cleaned.loc[failed].dropna().astype(str).unique().tolist())[:10]
        raise ValueError(
            f"{dataset}.{SNAPSHOT_COLUMN} contains missing/invalid dates: {examples}"
        )
    return pd.Series(parsed.to_numpy(), index=series.index, name=SNAPSHOT_COLUMN)


# ФУНКЦИЯ `_constant_reason`
# Определяет, является ли столбец полностью пустым или константным в конкретном split. Возвращаемая причина
# используется для симметричного удаления поля.
# Входы: `series`, `dataset`.
# Результат: значение типа `str | None`.
def _constant_reason(series: pd.Series, dataset: str) -> str | None:
    if series.isna().all():
        return f"all_missing_{dataset}"
    if series.nunique(dropna=False) <= 1:
        return f"constant_{dataset}"
    return None


# ФУНКЦИЯ `_assert_unique_columns`
# Ищет повторяющиеся имена столбцов и останавливает выполнение при неоднозначной схеме. Это предотвращает
# обращение к неправильной серии по имени.
# Входы: `frame`, `name`.
# Результат: значение типа `None`.
def _assert_unique_columns(frame: pd.DataFrame, name: str) -> None:
    duplicates = frame.columns[frame.columns.duplicated()].tolist()
    if duplicates:
        raise ValueError(f"{name} has duplicate columns: {duplicates}")


# ФУНКЦИЯ `parse_raw_frames`
# Центральный schema-contract: проверяет служебные поля и порядок train/test, строго парсит ID, даты, target,
# веса и числа, удаляет константы, исключает leakage и возвращает ParsedRawData. Ни одна модель не
# запускается, пока все инварианты не выполнены.
# Входы: `train`, `test`, `strict_numeric`, `require_all_special_columns`.
# Результат: значение типа `ParsedRawData`.
def parse_raw_frames(
    train: pd.DataFrame,
    test: pd.DataFrame,
    *,
    strict_numeric: bool = True,
    require_all_special_columns: bool = True,
) -> ParsedRawData:
    """Разбирает исходные таблицы и отделяет служебные столбцы от признаков."""
    train = train.copy()
    test = test.copy()
    _assert_unique_columns(train, "train")
    _assert_unique_columns(test, "test")
    required_train = {ID_COLUMN, SNAPSHOT_COLUMN, TARGET_COLUMN, WEIGHT_COLUMN}
    required_test = {ID_COLUMN, SNAPSHOT_COLUMN}
    if not required_train.issubset(train.columns):
        raise ValueError(
            f"train is missing service columns: {sorted(required_train - set(train))}"
        )
    if not required_test.issubset(test.columns):
        raise ValueError(
            f"test is missing service columns: {sorted(required_test - set(test))}"
        )
    leaked_test = {TARGET_COLUMN, WEIGHT_COLUMN}.intersection(test.columns)
    if leaked_test:
        raise ValueError(
            f"test unexpectedly contains label-only columns: {sorted(leaked_test)}"
        )
    train_feature_columns = [column for column in train if column not in required_train]
    test_feature_columns = [column for column in test if column not in required_test]
    if train_feature_columns != test_feature_columns:
        raise ValueError(
            f"raw train/test feature names and order differ; train-only={sorted(set(train_feature_columns) - set(test_feature_columns))[:10]}, test-only={sorted(set(test_feature_columns) - set(train_feature_columns))[:10]}"
        )
    if require_all_special_columns:
        missing_special = [
            column
            for column in SPECIAL_FEATURE_COLUMNS
            if column not in train_feature_columns
        ]
        if missing_special:
            raise ValueError(
                f"raw schema is missing declared special columns: {missing_special}"
            )
    parse_records: list[dict] = []
    for dataset_name, frame in (("train", train), ("test", test)):
        parsed_id, id_record = _numeric_parse(
            frame[ID_COLUMN], dataset=dataset_name, column=ID_COLUMN, kind="service_integer"
        )
        parse_records.append(id_record)
        if (
            parsed_id.isna().any()
            or np.isinf(parsed_id).any()
            or (not np.equal(parsed_id, np.floor(parsed_id)).all())
        ):
            raise ValueError(f"{dataset_name}.id must contain finite integers")
        frame[ID_COLUMN] = parsed_id.astype(np.int64)
        frame[SNAPSHOT_COLUMN] = _parse_snapshot_dates(
            frame[SNAPSHOT_COLUMN], dataset=dataset_name
        )
    for column in (TARGET_COLUMN, WEIGHT_COLUMN):
        parsed, record = _numeric_parse(
            train[column], dataset="train", column=column, kind="service_numeric"
        )
        parse_records.append(record)
        train[column] = parsed
    numeric_columns = [
        column for column in train_feature_columns if column not in SPECIAL_FEATURE_COLUMNS
    ]
    for column in numeric_columns:
        for dataset_name, frame in (("train", train), ("test", test)):
            parsed, record = _numeric_parse(
                frame[column], dataset=dataset_name, column=column, kind="feature_numeric"
            )
            frame[column] = parsed
            parse_records.append(record)
    for column in SPECIAL_FEATURE_COLUMNS:
        if column in train_feature_columns:
            train[column] = _clean_string_series(train[column])
            test[column] = _clean_string_series(test[column])
    parse_report = pd.DataFrame.from_records(parse_records)
    failures = parse_report.loc[
        (parse_report["failed_count"] > 0) | (parse_report["infinity_count"] > 0)
    ]
    if strict_numeric and (not failures.empty):
        details = failures[
            ["dataset", "column", "failed_count", "infinity_count", "unexpected_tokens"]
        ]
        raise ValueError(
            f"numeric parsing found unexpected tokens or infinity:\n{details.to_string(index=False)}"
        )
    if train[[TARGET_COLUMN, WEIGHT_COLUMN]].isna().any().any():
        raise ValueError("target and weights must not be missing")
    y = train[TARGET_COLUMN].to_numpy(dtype=np.float64)
    w = train[WEIGHT_COLUMN].to_numpy(dtype=np.float64)
    if (
        not np.isfinite(y).all()
        or not np.isfinite(w).all()
        or (w < 0).any()
        or (w.sum() <= 0)
    ):
        raise ValueError(
            "target must be finite; weights must be finite, non-negative and non-zero"
        )
    train_ids = train[ID_COLUMN].copy()
    test_ids = test[ID_COLUMN].copy()
    if not train_ids.is_unique or not test_ids.is_unique:
        raise ValueError("id must be unique within train and test")
    overlap = np.intersect1d(train_ids.to_numpy(), test_ids.to_numpy())
    if len(overlap):
        raise ValueError(f"train/test id sets overlap, for example {overlap[:10].tolist()}")
    drop_records: list[dict] = []
    dropped: list[str] = []
    for column in train_feature_columns:
        train_reason = _constant_reason(train[column], "train")
        test_reason = _constant_reason(test[column], "test")
        reasons = [reason for reason in (train_reason, test_reason) if reason is not None]
        if reasons:
            dropped.append(column)
            drop_records.append({"column": column, "reason": "+".join(reasons)})
    if (
        "first_salary_income" in train_feature_columns
        and "first_salary_income" not in dropped
    ):
        raise ValueError(
            "first_salary_income was expected to be constant/all-missing in test; review the input version before changing the frozen schema"
        )
    retained = [column for column in train_feature_columns if column not in dropped]
    train_features = train.loc[:, retained].copy()
    test_features = test.loc[:, retained].copy()
    assert_raw_feature_frames(train_features, test_features)
    train_dates = train[SNAPSHOT_COLUMN].copy()
    test_dates = test[SNAPSHOT_COLUMN].copy()
    return ParsedRawData(
        train_features=train_features,
        test_features=test_features,
        y=y,
        w=w,
        train_ids=train_ids,
        test_ids=test_ids,
        train_dates=train_dates,
        test_dates=test_dates,
        train_month=month_key(train_dates),
        test_month=month_key(test_dates),
        dropped_columns=tuple(dropped),
    )


# ФУНКЦИЯ `load_raw_data`
# Читает train.csv и test.csv с заданными разделителем, кодировкой, decimal и NA-правилами, после чего
# передаёт таблицы в единый строгий parser.
# Входы: `data_dir`, `train_name`, `test_name`, `encoding`, `strict_numeric`, `force_all_string`.
# Результат: значение типа `ParsedRawData`.
def load_raw_data(
    data_dir: str | Path,
    *,
    train_name: str = "train.csv",
    test_name: str = "test.csv",
    encoding: str = "utf-8",
    strict_numeric: bool = True,
    force_all_string: bool = False,
) -> ParsedRawData:
    """Читает train/test CSV и применяет общий парсер."""
    data_dir = Path(data_dir)
    read_kwargs = {
        "sep": ";",
        "dtype": "string",
        "encoding": encoding,
        "keep_default_na": True,
        "na_values": list(RAW_NA_VALUES),
        "low_memory": False,
        "decimal": ",",
    }
    if force_all_string:
        read_kwargs["dtype"] = "string"
    else:
        read_kwargs["dtype"] = {
            SNAPSHOT_COLUMN: "string",
            **{column: "string" for column in SPECIAL_FEATURE_COLUMNS},
        }
    train = pd.read_csv(data_dir / train_name, **read_kwargs)
    test = pd.read_csv(data_dir / test_name, **read_kwargs)
    return parse_raw_frames(train, test, strict_numeric=strict_numeric)


# ФУНКЦИЯ `assert_raw_feature_frames`
# Проверяет уже очищенные raw-матрицы: одинаковый порядок столбцов, отсутствие service columns, корректные
# числовые dtype и отсутствие infinity.
# Входы: `train`, `test`.
# Результат: значение типа `None`.
def assert_raw_feature_frames(train: pd.DataFrame, test: pd.DataFrame) -> None:
    """Проверяет одинаковый порядок и типы исходных признаков train/test."""
    _assert_unique_columns(train, "raw train features")
    _assert_unique_columns(test, "raw test features")
    if list(train.columns) != list(test.columns):
        raise ValueError("raw train/test feature order differs")
    leaked = FORBIDDEN_FEATURE_COLUMNS.intersection(train.columns)
    if leaked:
        raise ValueError(
            f"service columns leaked into raw feature matrix: {sorted(leaked)}"
        )
    for name, frame in (("train", train), ("test", test)):
        numeric = [column for column in frame if column not in SPECIAL_FEATURE_COLUMNS]
        non_numeric = [
            column for column in numeric if not is_numeric_dtype(frame[column].dtype)
        ]
        if non_numeric:
            raise TypeError(f"{name} has unparsed numeric columns: {non_numeric[:10]}")
        if numeric:
            values = frame[numeric].to_numpy(dtype=np.float64, na_value=np.nan)
            if np.isinf(values).any():
                raise ValueError(f"{name} raw features contain infinity")


# ФУНКЦИЯ `_coerce_snapshot_dates`
# Синхронизирует даты со строковым индексом матрицы, проверяет длину и валидность, затем возвращает datetime-
# Series с тем же индексом.
# Входы: `snapshot_dates`, `index`.
# Результат: значение типа `pd.Series`.
def _coerce_snapshot_dates(snapshot_dates, index: pd.Index) -> pd.Series:
    if len(snapshot_dates) != len(index):
        raise ValueError("snapshot_dates length differs from the feature frame")
    parsed = pd.to_datetime(
        pd.Series(snapshot_dates).reset_index(drop=True), errors="coerce"
    )
    if parsed.isna().any():
        raise ValueError("snapshot_dates contain missing or invalid values")
    return pd.Series(parsed.to_numpy(), index=index)


# ФУНКЦИЯ `_numeric_series`
# Получает один raw-столбец как float64 без тихого подавления ошибок. Используется внутри feature engineering
# после прохождения строгого parser.
# Входы: `frame`, `column`.
# Результат: значение типа `pd.Series`.
def _numeric_series(frame: pd.DataFrame, column: str) -> pd.Series:
    parsed = pd.to_numeric(frame[column], errors="raise")
    return pd.Series(
        parsed.to_numpy(dtype=np.float64, na_value=np.nan), index=frame.index, name=column
    )


# ФУНКЦИЯ `_category_keys`
# Строит канонические ключи категорий через normalize_text и отдельный токен пропуска. Эти ключи используются
# для частот, rare и unknown.
# Входы: `series`.
# Результат: значение типа `pd.Series`.
def _category_keys(series: pd.Series) -> pd.Series:
    return normalize_text(series).fillna(MISSING_TOKEN)


# ФУНКЦИЯ `_region_parts`
# Разбирает структурированную строку регионов в каноническую комбинацию, уникальные числовые коды и флаги
# invalid/missing. Сохраняет порядок первого появления кодов.
# Входы: `value`.
# Результат: значение типа `tuple[str, list[int], bool, bool]`.
def _region_parts(value) -> tuple[str, list[int], bool, bool]:
    if pd.isna(value) or not str(value).strip():
        return (MISSING_TOKEN, [], False, True)
    text = str(value).strip()
    raw_codes = re.findall("\\d+", text)
    residue = re.sub("[\\d\\s,;|/\\\\\\[\\](){}]+", "", text)
    invalid = bool(residue) or not raw_codes
    codes: list[int] = []
    seen: set[int] = set()
    for token in raw_codes:
        code = int(token)
        if code not in seen:
            seen.add(code)
            codes.append(code)
    canonical = ",".join((str(code) for code in codes)) if codes else UNKNOWN_TOKEN
    return (canonical, codes, invalid, False)


In [5]:
# МОДУЛЬ 3. Target-free генератор признаков F0-F4.
# Обучает состояние преобразований только на доступном прошлом и одинаково применяет его к valid/test.

# КЛАСС `TargetFreeFeatureTransformer`
# Главный генератор признаков F0–F4 без доступа к target. На каждом временном fold отдельно изучает только
# частоты/уровни train и тем же состоянием преобразует valid/test.
# Входы: поля экземпляра и методы, объявленные ниже.
# Результат: экземпляр класса, используемый как явный контракт состояния.
class TargetFreeFeatureTransformer:
    """Трансформер признаков без использования target. Категории и частоты обучаются отдельно внутри каждого временного фолда."""

    # ФУНКЦИЯ `TargetFreeFeatureTransformer.__init__`
    # Сохраняет переданную FeatureConfig либо создаёт frozen-конфигурацию по умолчанию. Обучаемое состояние
    # появится только после fit.
    # Входы: `config`.
    # Результат: `None`; функция выполняет проверку или изменяет подготовленное состояние.
    def __init__(self, config: FeatureConfig | None = None):
        self.config = config or FeatureConfig()

    # ФУНКЦИЯ `TargetFreeFeatureTransformer._validate_frame`
    # Проверяет тип DataFrame, уникальность и порядок столбцов, отсутствие target/service columns, числовые
    # dtype и infinity. В режиме transform требует точного совпадения с fitted schema.
    # Входы: `frame`, `fitting`.
    # Результат: значение типа `None`.
    def _validate_frame(self, frame: pd.DataFrame, *, fitting: bool) -> None:
        if not isinstance(frame, pd.DataFrame):
            raise TypeError("features must be a pandas DataFrame")
        _assert_unique_columns(frame, "feature frame")
        leaked = FORBIDDEN_FEATURE_COLUMNS.intersection(frame.columns)
        if leaked:
            raise ValueError(
                f"target/service columns are forbidden in feature engineering: {sorted(leaked)}"
            )
        if fitting:
            if not len(frame):
                raise ValueError("cannot fit the transformer on an empty frame")
        elif tuple(frame.columns) != self.input_columns_:
            raise ValueError(
                "transform input names/order differ from the fitted raw schema"
            )
        numeric = [column for column in frame if column not in SPECIAL_FEATURE_COLUMNS]
        bad = [column for column in numeric if not is_numeric_dtype(frame[column].dtype)]
        if bad:
            raise TypeError(f"numeric raw columns have non-numeric dtype: {bad[:10]}")
        if numeric:
            values = frame[numeric].to_numpy(dtype=np.float64, na_value=np.nan)
            if np.isinf(values).any():
                raise ValueError("raw feature frame contains infinity")

    # ФУНКЦИЯ `TargetFreeFeatureTransformer.fit`
    # Изучает только распределение доступного train-fold: фиксирует raw-схему, частоты категорий,
    # frequent/rare уровни и статистику регионов. Target и веса функция принципиально не принимает.
    # Входы: `frame`, `snapshot_dates`.
    # Результат: значение типа `'TargetFreeFeatureTransformer'`.
    def fit(self, frame: pd.DataFrame, snapshot_dates) -> "TargetFreeFeatureTransformer":
        self._validate_frame(frame, fitting=True)
        _coerce_snapshot_dates(snapshot_dates, frame.index)
        self.input_columns_ = tuple(frame.columns)
        self.numeric_columns_ = tuple(
            (column for column in frame if column not in SPECIAL_FEATURE_COLUMNS)
        )
        unknown_exclusions = sorted(
            set(self.config.excluded_raw_columns) - set(self.numeric_columns_)
        )
        if unknown_exclusions:
            raise ValueError(
                f"excluded_raw_columns are absent or non-numeric: {unknown_exclusions}"
            )
        self.category_stats_: dict[str, dict] = {}
        distribution_columns = (*CATEGORY_COLUMNS, *HIGH_CARD_TEXT_COLUMNS)
        threshold = max(
            self.config.rare_min_count,
            int(math.ceil(self.config.rare_min_fraction * len(frame))),
        )
        for column in distribution_columns:
            if column not in frame:
                continue
            keys = _category_keys(frame[column])
            raw_counts = keys.value_counts(dropna=False, sort=False)
            counts = {str(key): int(value) for key, value in raw_counts.items()}
            frequent = tuple(
                sorted(
                    (
                        key
                        for key, count in counts.items()
                        if key != MISSING_TOKEN and count >= threshold
                    )
                )
            )
            self.category_stats_[column] = {
                "counts": counts,
                "frequent": frequent,
                "threshold": threshold,
                "denominator": len(frame),
                "native_levels": (MISSING_TOKEN, RARE_TOKEN, UNKNOWN_TOKEN, *frequent),
            }
        self.region_stats_: dict | None = None
        if STRUCTURED_COLUMNS[0] in frame:
            canonical = frame[STRUCTURED_COLUMNS[0]].map(
                lambda value: _region_parts(value)[0]
            )
            raw_counts = canonical.value_counts(dropna=False, sort=False)
            counts = {str(key): int(value) for key, value in raw_counts.items()}
            frequent = tuple(
                sorted(
                    (
                        key
                        for key, count in counts.items()
                        if key != MISSING_TOKEN and count >= threshold
                    )
                )
            )
            self.region_stats_ = {
                "counts": counts,
                "frequent": frequent,
                "threshold": threshold,
                "denominator": len(frame),
            }
        self.numeric_feature_names_out_: tuple[str, ...] | None = None
        self.lightgbm_feature_names_out_: tuple[str, ...] | None = None
        return self

    # ФУНКЦИЯ `TargetFreeFeatureTransformer._mapped_category`
    # Отображает канонический ключ в одно из состояний: missing, unknown, rare или сохранённая частая
    # категория. Возвращает Series с замороженными правилами fold.
    # Входы: `keys`, `stats`.
    # Результат: значение типа `pd.Series`.
    @staticmethod
    def _mapped_category(keys: pd.Series, stats: dict) -> pd.Series:
        seen = frozenset(stats["counts"])
        frequent = frozenset(stats["frequent"])

        # ФУНКЦИЯ `TargetFreeFeatureTransformer._mapped_category.map_value`
        # Классифицирует одно категориальное значение относительно seen/frequent множеств текущего fold. Эта
        # вложенная функция применяется ко всей Series.
        # Входы: `value`.
        # Результат: значение типа `str`.
        def map_value(value: str) -> str:
            if value == MISSING_TOKEN:
                return MISSING_TOKEN
            if value not in seen:
                return UNKNOWN_TOKEN
            if value not in frequent:
                return RARE_TOKEN
            return value

        return keys.map(map_value)

    # ФУНКЦИЯ `TargetFreeFeatureTransformer._add_category_features`
    # Добавляет frequency, rare/unknown/missing, one-hot, текстовые и региональные признаки; при необходимости
    # создаёт native categorical columns с frozen levels для LightGBM.
    # Входы: `result`, `frame`, `include_native_categories`.
    # Результат: значение типа `None`.
    def _add_category_features(
        self, result: pd.DataFrame, frame: pd.DataFrame, *, include_native_categories: bool
    ) -> None:
        for column in (*CATEGORY_COLUMNS, *HIGH_CARD_TEXT_COLUMNS):
            if column not in self.category_stats_:
                continue
            stats = self.category_stats_[column]
            keys = _category_keys(frame[column])
            mapped = self._mapped_category(keys, stats)
            frequency = (
                keys.map(stats["counts"]).fillna(0).astype(np.float64)
                / stats["denominator"]
            )
            result[f"cat_frequency__{column}"] = frequency.astype(np.float32)
            result[f"cat_rare__{column}"] = mapped.eq(RARE_TOKEN).to_numpy(dtype=np.int8)
            result[f"cat_unknown__{column}"] = mapped.eq(UNKNOWN_TOKEN).to_numpy(
                dtype=np.int8
            )
            result[f"cat_missing__{column}"] = mapped.eq(MISSING_TOKEN).to_numpy(
                dtype=np.int8
            )
            if column in self.config.one_hot_columns:
                for position, value in enumerate(stats["frequent"]):
                    result[f"cat_onehot__{column}__{position:03d}"] = keys.eq(
                        value
                    ).to_numpy(dtype=np.int8)
            if column in HIGH_CARD_TEXT_COLUMNS:
                normalized = normalize_text(frame[column])
                result[f"text_length__{column}"] = normalized.str.len().to_numpy(
                    dtype=np.float32, na_value=np.nan
                )
                result[f"text_word_count__{column}"] = normalized.str.count(
                    "[\\w-]+"
                ).to_numpy(dtype=np.float32, na_value=np.nan)
                result[f"text_has_digit__{column}"] = normalized.str.contains(
                    "\\d", regex=True, na=False
                ).to_numpy(dtype=np.int8)
            if include_native_categories and column in CATEGORY_COLUMNS:
                dtype = pd.CategoricalDtype(
                    categories=list(stats["native_levels"]), ordered=False
                )
                result[f"nativecat__{column}"] = mapped.astype(dtype)
        address_column = STRUCTURED_COLUMNS[0]
        if self.region_stats_ is not None and address_column in frame:
            parts = frame[address_column].map(_region_parts)
            canonical = parts.map(lambda value: value[0])
            code_lists = parts.map(lambda value: value[1])
            invalid = parts.map(lambda value: value[2])
            missing = parts.map(lambda value: value[3])
            seen = frozenset(self.region_stats_["counts"])
            frequent = frozenset(self.region_stats_["frequent"])
            result["address_region_count"] = code_lists.map(len).astype(np.int16)
            first = code_lists.map(lambda values: float(values[0]) if values else np.nan)
            result["address_region_first"] = first.astype(np.float32)
            result["address_region_multi"] = code_lists.map(
                lambda values: len(values) > 1
            ).to_numpy(dtype=np.int8)
            result["address_region_invalid"] = invalid.to_numpy(dtype=np.int8)
            result["address_region_missing"] = missing.to_numpy(dtype=np.int8)
            result["address_region_frequency"] = (
                canonical.map(self.region_stats_["counts"]).fillna(0).astype(np.float64)
                / self.region_stats_["denominator"]
            ).astype(np.float32)
            result["address_region_rare"] = canonical.map(
                lambda value: value not in {MISSING_TOKEN, UNKNOWN_TOKEN}
                and value in seen
                and (value not in frequent)
            ).to_numpy(dtype=np.int8)
            result["address_region_unknown"] = canonical.map(
                lambda value: value == UNKNOWN_TOKEN or value not in seen
            ).to_numpy(dtype=np.int8)

    # ФУНКЦИЯ `TargetFreeFeatureTransformer._add_period_features`
    # Разбирает period_last_act_ad относительно даты снимка и создаёт возраст активности, missing, invalid и
    # future flags. Sentinel 1677 не трактуется как настоящая дата.
    # Входы: `result`, `frame`, `snapshot_dates`.
    # Результат: значение типа `None`.
    @staticmethod
    def _add_period_features(
        result: pd.DataFrame, frame: pd.DataFrame, snapshot_dates: pd.Series
    ) -> None:
        column = DATE_COLUMNS[0]
        if column not in frame:
            return
        cleaned = _clean_string_series(frame[column])
        years = pd.to_numeric(
            cleaned.str.extract("^(\\d{4})", expand=False), errors="coerce"
        )
        sentinel = years.eq(1677).fillna(False)
        parsed = pd.to_datetime(cleaned.mask(sentinel), format="%Y-%m-%d", errors="coerce")
        missing = cleaned.isna()
        invalid = ~missing & (sentinel | parsed.isna())
        future = parsed.notna() & parsed.gt(snapshot_dates)
        snapshot_index = snapshot_dates.dt.year * 12 + snapshot_dates.dt.month
        activity_index = parsed.dt.year * 12 + parsed.dt.month
        months_since = snapshot_index - activity_index
        months_since = months_since.mask(invalid | future)
        result["months_since_last_activity"] = months_since.to_numpy(
            dtype=np.float32, na_value=np.nan
        )
        result["last_activity_missing"] = missing.to_numpy(dtype=np.int8)
        result["last_activity_invalid"] = invalid.to_numpy(dtype=np.int8)
        result["last_activity_future"] = future.to_numpy(dtype=np.int8)

    # ФУНКЦИЯ `TargetFreeFeatureTransformer._add_missingness_features`
    # Превращает структуру пропусков в признаки: общее число/долю, доступность смысловых семейств и отдельные
    # high-signal missing flags.
    # Входы: `result`, `frame`.
    # Результат: значение типа `None`.
    def _add_missingness_features(self, result: pd.DataFrame, frame: pd.DataFrame) -> None:
        missing = frame.isna()
        result["row_missing_count"] = missing.sum(axis=1).astype(np.int16)
        result["row_missing_fraction"] = missing.mean(axis=1).astype(np.float32)
        for family, prefixes in SOURCE_FAMILY_PREFIXES.items():
            columns = [
                column for column in self.numeric_columns_ if column.startswith(prefixes)
            ]
            if columns:
                result[f"missing_count__{family}"] = (
                    frame[columns].isna().sum(axis=1).astype(np.int16)
                )
                result[f"available_count__{family}"] = (
                    frame[columns].notna().sum(axis=1).astype(np.int16)
                )
        for column in self.config.high_signal_missing_columns:
            if column in frame:
                result[f"missing__{column}"] = frame[column].isna().to_numpy(dtype=np.int8)

    # ФУНКЦИЯ `TargetFreeFeatureTransformer._safe_ratio`
    # Безопасно вычисляет отношение двух величин только при существующем ненулевом знаменателе и конечном
    # результате. Дополнительно создаёт флаг валидности ratio.
    # Входы: `result`, `frame`, `name`, `numerator`, `denominator`, `numerator_override`,
    # `denominator_override`.
    # Результат: значение типа `None`.
    @staticmethod
    def _safe_ratio(
        result: pd.DataFrame,
        frame: pd.DataFrame,
        name: str,
        numerator: str,
        denominator: str,
        *,
        numerator_override: pd.Series | None = None,
        denominator_override: pd.Series | None = None,
    ) -> None:
        if numerator_override is None and numerator not in frame:
            return
        if denominator_override is None and denominator not in frame:
            return
        num = (
            numerator_override
            if numerator_override is not None
            else _numeric_series(frame, numerator)
        )
        den = (
            denominator_override
            if denominator_override is not None
            else _numeric_series(frame, denominator)
        )
        valid = num.notna() & den.notna() & den.ne(0)
        ratio = pd.Series(np.nan, index=frame.index, dtype=np.float64)
        ratio.loc[valid] = num.loc[valid] / den.loc[valid]
        finite = (
            np.isfinite(ratio.to_numpy(dtype=np.float64, na_value=np.nan))
            | ratio.isna().to_numpy()
        )
        valid &= finite
        ratio = ratio.where(valid)
        result[name] = ratio.astype(np.float32)
        result[f"valid__{name}"] = valid.to_numpy(dtype=np.int8)

    # ФУНКЦИЯ `TargetFreeFeatureTransformer._add_financial_features`
    # Строит робастные агрегаты денежных proxy, паттерн доступности и финансовые отношения. Отрицательные
    # proxy маскируются как недействительные.
    # Входы: `result`, `frame`.
    # Результат: значение типа `None`.
    def _add_financial_features(self, result: pd.DataFrame, frame: pd.DataFrame) -> None:
        income_columns = [column for column in INCOME_PROXY_COLUMNS if column in frame]
        if income_columns:
            proxy = pd.DataFrame(
                {column: _numeric_series(frame, column) for column in income_columns},
                index=frame.index,
            ).mask(lambda values: values < 0)
            median = proxy.median(axis=1, skipna=True)
            result["income_proxy_median"] = median.astype(np.float32)
            result["income_proxy_mean"] = proxy.mean(axis=1, skipna=True).astype(np.float32)
            result["income_proxy_max"] = proxy.max(axis=1, skipna=True).astype(np.float32)
            result["income_proxy_min"] = proxy.min(axis=1, skipna=True).astype(np.float32)
            result["income_proxy_spread"] = (
                result["income_proxy_max"] - result["income_proxy_min"]
            ).astype(np.float32)
            available = proxy.notna()
            result["income_proxy_available_count"] = available.sum(axis=1).astype(np.int8)
            result["income_proxy_positive_count"] = proxy.gt(0).sum(axis=1).astype(np.int8)
            pattern = np.zeros(len(frame), dtype=np.int32)
            for position, column in enumerate(income_columns):
                pattern |= available[column].to_numpy(dtype=np.int32) << position
            result["income_proxy_availability_pattern"] = pattern
            self._safe_ratio(
                result,
                frame,
                "salary_to_income_proxy_median",
                "salary_6to12m_avg",
                "income_proxy_median",
                denominator_override=median,
            )
            self._safe_ratio(
                result,
                frame,
                "declared_to_income_proxy_median",
                "incomeValue",
                "income_proxy_median",
                denominator_override=median,
            )
            self._safe_ratio(
                result,
                frame,
                "debt_to_income_proxy_median",
                "hdb_outstand_sum",
                "income_proxy_median",
                denominator_override=median,
            )
            self._safe_ratio(
                result,
                frame,
                "loan_to_income_proxy_median",
                "loan_cur_amt",
                "income_proxy_median",
                denominator_override=median,
            )
        self._safe_ratio(
            result,
            frame,
            "credit_to_debit_turn",
            "turn_cur_cr_avg_v2",
            "turn_cur_db_avg_v2",
        )
        self._safe_ratio(
            result,
            frame,
            "debt_to_total_limit",
            "hdb_outstand_sum",
            "hdb_bki_total_max_limit",
        )

    # ФУНКЦИЯ `TargetFreeFeatureTransformer.transform`
    # Собирает итоговую матрицу в фиксированном порядке F0→F4, при необходимости добавляет snapshot month,
    # фиксирует output schema и проверяет модельные dtype/infinity.
    # Входы: `frame`, `snapshot_dates`, `include_native_categories`.
    # Результат: значение типа `pd.DataFrame`.
    def transform(
        self,
        frame: pd.DataFrame,
        snapshot_dates,
        *,
        include_native_categories: bool = False,
    ) -> pd.DataFrame:
        if not hasattr(self, "input_columns_"):
            raise RuntimeError("transformer must be fitted before transform")
        self._validate_frame(frame, fitting=False)
        dates = _coerce_snapshot_dates(snapshot_dates, frame.index)
        result = pd.DataFrame(index=frame.index)
        if self.config.include_f0_raw:
            excluded = frozenset(self.config.excluded_raw_columns)
            result = pd.DataFrame(
                {
                    column: _numeric_series(frame, column).astype(np.float32)
                    for column in self.numeric_columns_
                    if column not in excluded
                },
                index=frame.index,
            )
        if self.config.include_f1_categories:
            self._add_category_features(
                result, frame, include_native_categories=include_native_categories
            )
        if self.config.include_f2_time:
            self._add_period_features(result, frame, dates)
            if self.config.include_snapshot_month:
                result["snapshot_month_index"] = (
                    dates.dt.year * 12 + dates.dt.month
                ).to_numpy(dtype=np.int32)
        if self.config.include_f3_missingness:
            self._add_missingness_features(result, frame)
        if self.config.include_f4_financial:
            self._add_financial_features(result, frame)
        if not len(result.columns):
            raise ValueError("feature configuration produced an empty matrix")
        expected_attribute = (
            "lightgbm_feature_names_out_"
            if include_native_categories
            else "numeric_feature_names_out_"
        )
        expected = getattr(self, expected_attribute)
        current = tuple(result.columns)
        if expected is None:
            setattr(self, expected_attribute, current)
        elif current != expected:
            raise RuntimeError("engineered feature order changed between transform calls")
        assert_model_feature_frame(
            result, allow_native_categories=include_native_categories
        )
        return result

    # ФУНКЦИЯ `TargetFreeFeatureTransformer.fit_transform`
    # Последовательно выполняет fit и transform на одной учебной части. Это сокращённый безопасный путь для
    # train-fold.
    # Входы: `frame`, `snapshot_dates`, `include_native_categories`.
    # Результат: значение типа `pd.DataFrame`.
    def fit_transform(
        self,
        frame: pd.DataFrame,
        snapshot_dates,
        *,
        include_native_categories: bool = False,
    ) -> pd.DataFrame:
        return self.fit(frame, snapshot_dates).transform(
            frame, snapshot_dates, include_native_categories=include_native_categories
        )

    # ФУНКЦИЯ `TargetFreeFeatureTransformer.get_feature_names_out`
    # Возвращает уже зафиксированный порядок выходных признаков для numeric либо native-categorical режима. До
    # первого transform вызов запрещён.
    # Входы: `include_native_categories`.
    # Результат: значение типа `tuple[str, ...]`.
    def get_feature_names_out(
        self, *, include_native_categories: bool = False
    ) -> tuple[str, ...]:
        attribute = (
            "lightgbm_feature_names_out_"
            if include_native_categories
            else "numeric_feature_names_out_"
        )
        if not hasattr(self, attribute) or getattr(self, attribute) is None:
            raise RuntimeError(
                "call transform for this output mode before requesting feature names"
            )
        return getattr(self, attribute)


# ФУНКЦИЯ `assert_model_feature_frame`
# Проверяет готовую модельную матрицу: уникальные имена, отсутствие leakage, допустимые numeric/category
# dtype, отсутствие infinity и повторов уровней категорий.
# Входы: `frame`, `allow_native_categories`.
# Результат: значение типа `None`.
def assert_model_feature_frame(
    frame: pd.DataFrame, *, allow_native_categories: bool = False
) -> None:
    """Проверяет матрицу модели; числовые NaN разрешены."""
    _assert_unique_columns(frame, "model feature frame")
    leaked = FORBIDDEN_FEATURE_COLUMNS.intersection(frame.columns)
    if leaked:
        raise ValueError(f"service columns leaked into model features: {sorted(leaked)}")
    numeric_columns: list[str] = []
    category_columns: list[str] = []
    unsupported: list[str] = []
    for column in frame:
        dtype = frame[column].dtype
        if is_numeric_dtype(dtype):
            numeric_columns.append(column)
        elif isinstance(dtype, pd.CategoricalDtype) and allow_native_categories:
            category_columns.append(column)
        else:
            unsupported.append(column)
    if unsupported:
        raise TypeError(f"unsupported model feature dtypes: {unsupported[:10]}")
    if numeric_columns:
        values = frame[numeric_columns].to_numpy(dtype=np.float64, na_value=np.nan)
        if np.isinf(values).any():
            raise ValueError("model matrix contains infinity")
    for column in category_columns:
        if frame[column].cat.categories.has_duplicates:
            raise ValueError(f"native category {column} has duplicate levels")


In [6]:
# МОДУЛЬ 4. Загрузка данных и первичный EDA-контроль.
# Читает три входных CSV, проверяет совместимость train/test и фиксирует фактические размеры задачи.

raw = load_raw_data(DATA_DIR, strict_numeric=True)
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", sep=";")

if raw.train_features.shape[1] != raw.test_features.shape[1]:
    raise RuntimeError("Наборы признаков train/test различаются")
if sample_submission.columns.tolist() != ["id", "predict"]:
    raise RuntimeError("Ожидались столбцы id и predict в sample_submission.csv")
if len(sample_submission) != len(raw.test_ids):
    raise RuntimeError("Размер sample_submission.csv не совпадает с test.csv")
if not sample_submission["id"].is_unique:
    raise RuntimeError("В sample_submission.csv есть повторяющиеся id")
if not np.array_equal(sample_submission["id"].to_numpy(), raw.test_ids.to_numpy()):
    raise RuntimeError("Порядок id в sample_submission.csv не совпадает с test.csv")

summary = pd.DataFrame(
    {
        "split": ["train", "test"],
        "rows": [len(raw.train_features), len(raw.test_features)],
        "features": [raw.train_features.shape[1], raw.test_features.shape[1]],
        "missing_fraction": [
            raw.train_features.isna().to_numpy().mean(),
            raw.test_features.isna().to_numpy().mean(),
        ],
    }
)
month_summary = (
    pd.DataFrame(
        {
            "split": ["train"] * len(raw.train_month) + ["test"] * len(raw.test_month),
            "month": np.concatenate([raw.train_month, raw.test_month]),
        }
    )
    .value_counts(sort=False)
    .rename("rows")
    .reset_index()
)

display(summary)
display(month_summary)
print(f"Удалено константных или пустых признаков: {len(raw.dropped_columns)}")


,split,rows,features,missing_fraction
0,train,76786,219,0.422044
1,test,73214,219,0.414153


,split,month,rows
0,train,24292,14858
1,train,24290,8865
2,train,24291,13413
3,train,24294,16214
4,train,24289,7243
5,train,24293,16193
6,test,24296,14646
7,test,24298,12598
8,test,24297,15501
9,test,24299,13674


Удалено константных или пустых признаков: 1


## 3. LightGBM и конфигурация моделей


In [7]:
# МОДУЛЬ 5. Базовые обёртки LightGBM и weighted blend.
# Содержит единый способ обучения компонентов, расчёта WMAE и объединения прогнозов V4.

SEED = 20260712


# КЛАСС `LightGBMConfig`
# Неизменяемое описание детерминированного CPU LightGBM: objective L1, sampling, regularization, seeds,
# binning и число потоков.
# Входы: поля экземпляра и методы, объявленные ниже.
# Результат: экземпляр класса, используемый как явный контракт состояния.
@dataclass(frozen=True)
class LightGBMConfig:
    """Параметры детерминированного CPU-обучения LightGBM."""

    num_threads: int = 4
    seed: int = SEED
    learning_rate: float = 0.05
    num_leaves: int = 31
    min_data_in_leaf: int = 50
    feature_fraction: float = 0.85
    bagging_fraction: float = 0.9
    bagging_freq: int = 1
    lambda_l1: float = 0.2
    lambda_l2: float = 3.0
    max_bin: int = 127
    extra_params: Mapping[str, Any] = field(default_factory=dict, repr=False, compare=False)

    # ФУНКЦИЯ `LightGBMConfig.__post_init__`
    # Проверяет минимально допустимые параметры конфигурации до дорогостоящего обучения. Некорректное число
    # потоков или max_bin вызывает раннюю ошибку.
    # Входы: нет явных аргументов.
    # Результат: значение типа `None`.
    def __post_init__(self) -> None:
        if self.num_threads < 1:
            raise ValueError("num_threads must be positive")
        if self.max_bin < 2:
            raise ValueError("max_bin must be at least 2")

    # ФУНКЦИЯ `LightGBMConfig.params`
    # Собирает словарь параметров для lgb.train, добавляет специальные настройки компонента и запрещает
    # переход на недетерминированный GPU-путь.
    # Входы: нет явных аргументов.
    # Результат: значение типа `dict[str, Any]`.
    def params(self) -> dict[str, Any]:
        params: dict[str, Any] = {
            "objective": "regression_l1",
            "metric": "l1",
            "device_type": "cpu",
            "deterministic": True,
            "force_col_wise": True,
            "num_threads": self.num_threads,
            "seed": self.seed,
            "data_random_seed": self.seed,
            "feature_fraction_seed": self.seed,
            "bagging_seed": self.seed,
            "verbosity": -1,
            "learning_rate": self.learning_rate,
            "num_leaves": self.num_leaves,
            "min_data_in_leaf": self.min_data_in_leaf,
            "feature_fraction": self.feature_fraction,
            "bagging_fraction": self.bagging_fraction,
            "bagging_freq": self.bagging_freq,
            "lambda_l1": self.lambda_l1,
            "lambda_l2": self.lambda_l2,
            "max_bin": self.max_bin,
        }
        params.update(dict(self.extra_params))
        if params.get("device_type") != "cpu":
            raise ValueError("the reproducible LightGBM path is CPU-only")
        return params


# ФУНКЦИЯ `_arrays`
# Проверяет label и weights для LightGBM: форму, длину относительно X, конечность и положительную сумму весов.
# Возвращает float64-векторы.
# Входы: `y`, `weights`, `expected_length`.
# Результат: значение типа `tuple[np.ndarray, np.ndarray]`.
def _arrays(
    y, weights, *, expected_length: int | None = None
) -> tuple[np.ndarray, np.ndarray]:
    y_array = np.asarray(y, dtype=np.float64)
    w_array = np.asarray(weights, dtype=np.float64)
    if (
        y_array.ndim != 1
        or w_array.ndim != 1
        or len(y_array) != len(w_array)
        or (not len(y_array))
    ):
        raise ValueError("labels and weights must be same-length non-empty vectors")
    if expected_length is not None and len(y_array) != expected_length:
        raise ValueError("labels/weights length differs from feature matrix")
    if not np.isfinite(y_array).all() or not np.isfinite(w_array).all():
        raise ValueError("labels and weights must be finite")
    if (w_array < 0).any() or w_array.sum(dtype=np.float64) <= 0:
        raise ValueError("weights must be non-negative with positive sum")
    return (y_array, w_array)


# КЛАСС `FittedLightGBM`
# Контейнер обученного booster вместе с точным порядком признаков, категориальными столбцами, уровнями и
# числом использованных деревьев.
# Входы: поля экземпляра и методы, объявленные ниже.
# Результат: экземпляр класса, используемый как явный контракт состояния.
@dataclass(frozen=True)
class FittedLightGBM:
    booster: Any = field(repr=False)
    feature_columns: tuple[str, ...]
    categorical_columns: tuple[str, ...]
    categorical_levels: tuple[tuple[Any, ...], ...]
    rounds: int


# ФУНКЦИЯ `_categorical_columns`
# Возвращает ordered tuple всех pandas categorical columns в матрице. Порядок входит в контракт обучения и
# prediction.
# Входы: `frame`.
# Результат: значение типа `tuple[str, ...]`.
def _categorical_columns(frame: pd.DataFrame) -> tuple[str, ...]:
    return tuple(
        (column for column in frame if isinstance(frame[column].dtype, pd.CategoricalDtype))
    )


# ФУНКЦИЯ `_categorical_levels`
# Сохраняет точный порядок уровней каждой категориальной колонки. Позволяет обнаружить несовместимую матрицу
# при прогнозе.
# Входы: `frame`, `columns`.
# Результат: значение типа `tuple[tuple[Any, ...], ...]`.
def _categorical_levels(
    frame: pd.DataFrame, columns: Sequence[str]
) -> tuple[tuple[Any, ...], ...]:
    return tuple((tuple(frame[column].cat.categories.tolist()) for column in columns))


# ФУНКЦИЯ `fit_lightgbm_fixed`
# Проверяет X/y/w, нормирует recency weights, создаёт lgb.Dataset и обучает CPU LightGBM ровно заданное число
# раундов. Возвращает booster вместе со schema-contract.
# Входы: `X_train`, `y_train`, `weights_train`, `train_months`, `rounds`, `decay`, `config`.
# Результат: значение типа `FittedLightGBM`.
def fit_lightgbm_fixed(
    X_train: pd.DataFrame,
    y_train,
    weights_train,
    train_months,
    *,
    rounds: int,
    decay: float = 1.0,
    config: LightGBMConfig | None = None,
) -> FittedLightGBM:
    import lightgbm as lgb

    config = config or LightGBMConfig()
    if rounds < 1:
        raise ValueError("rounds must be positive")
    assert_model_feature_frame(X_train, allow_native_categories=True)
    y_train, weights_train = _arrays(y_train, weights_train, expected_length=len(X_train))
    fit_weights = recency_weights(weights_train, train_months, decay)
    categorical = _categorical_columns(X_train)
    dataset = lgb.Dataset(
        X_train,
        label=y_train,
        weight=fit_weights,
        categorical_feature=list(categorical),
        free_raw_data=False,
    )
    booster = lgb.train(
        config.params(),
        dataset,
        num_boost_round=int(rounds),
        callbacks=[lgb.log_evaluation(0)],
        keep_training_booster=True,
    )
    return FittedLightGBM(
        booster,
        tuple(X_train.columns),
        categorical,
        _categorical_levels(X_train, categorical),
        int(rounds),
    )


# ФУНКЦИЯ `predict_lightgbm_fixed`
# Перед прогнозом сверяет столбцы, категории и уровни с обучением, затем вызывает booster на фиксированном
# числе итераций и запрещает non-finite результат.
# Входы: `model`, `X`.
# Результат: значение типа `np.ndarray`.
def predict_lightgbm_fixed(model: FittedLightGBM, X: pd.DataFrame) -> np.ndarray:
    assert_model_feature_frame(X, allow_native_categories=True)
    if tuple(X.columns) != model.feature_columns:
        raise ValueError("LightGBM prediction schema/order differs from training")
    if _categorical_columns(X) != model.categorical_columns:
        raise ValueError("LightGBM categorical schema differs from training")
    if _categorical_levels(X, model.categorical_columns) != model.categorical_levels:
        raise ValueError("LightGBM category levels/order differ from training")
    prediction = np.asarray(
        model.booster.predict(X, num_iteration=model.rounds, validate_features=True),
        dtype=np.float64,
    )
    if not np.isfinite(prediction).all():
        raise ValueError("LightGBM produced non-finite predictions")
    return prediction


# ФУНКЦИЯ `blend_component_predictions`
# Строит выпуклый именованный blend компонентов. Проверяет совпадение имён, сумму весов 1, формы и конечность
# массивов, затем складывает их в стабильном порядке.
# Входы: `predictions`, `weights`.
# Результат: значение типа `np.ndarray`.
def blend_component_predictions(
    predictions: Mapping[str, Sequence[float] | np.ndarray], weights: Mapping[str, float]
) -> np.ndarray:
    """Смешивает именованные компоненты и проверяет соответствие весов."""
    if not isinstance(predictions, Mapping) or not isinstance(weights, Mapping):
        raise TypeError("predictions and weights must be mappings")
    prediction_keys = set(predictions)
    weight_keys = set(weights)
    if not prediction_keys or prediction_keys != weight_keys:
        raise ValueError(
            "prediction/weight component names must be identical and non-empty"
        )
    numeric_weights = {name: float(weights[name]) for name in sorted(weight_keys)}
    if not all((np.isfinite(value) and value >= 0 for value in numeric_weights.values())):
        raise ValueError("component weights must be finite and non-negative")
    if not np.isclose(sum(numeric_weights.values()), 1.0, rtol=0, atol=1e-12):
        raise ValueError("component weights must sum to one")
    arrays: dict[str, np.ndarray] = {}
    expected_shape = None
    for name in sorted(prediction_keys):
        array = np.asarray(predictions[name], dtype=np.float64)
        if array.ndim != 1 or not len(array):
            raise ValueError(f"component {name!r} prediction must be a non-empty vector")
        if not np.isfinite(array).all():
            raise ValueError(f"component {name!r} prediction contains non-finite values")
        if expected_shape is None:
            expected_shape = array.shape
        elif array.shape != expected_shape:
            raise ValueError("component prediction shapes differ")
        arrays[name] = array
    blended = np.zeros(expected_shape, dtype=np.float64)
    for name in sorted(arrays):
        blended += numeric_weights[name] * arrays[name]
    if not np.isfinite(blended).all():
        raise ValueError("component blend produced non-finite predictions")
    return blended


In [8]:
# МОДУЛЬ 6. Замороженная конфигурация трёх компонентов V4.
# Фиксирует признаки, гиперпараметры, число раундов и веса tuned/wide/snapshot моделей.

SNAPSHOT_FEATURE_NAME = "snapshot_month_index"
FROZEN_ROUNDS_TUNED = 2300
FROZEN_ROUNDS_WIDE = 1100
FROZEN_ROUNDS_SNAPSHOT = 2600
BASE_TUNED_SHARE = 0.6
BASE_WIDE_SHARE = 0.4
STABLE_BLEND_WEIGHT = 0.6
SNAPSHOT_WEIGHT = 0.4
TUNED_WEIGHT = STABLE_BLEND_WEIGHT * BASE_TUNED_SHARE
WIDE_WEIGHT = STABLE_BLEND_WEIGHT * BASE_WIDE_SHARE


# ФУНКЦИЯ `final_stable_feature_config`
# Возвращает frozen-набор F0–F4 без абсолютного индекса месяца. Это stable-представление двух компонентов V4.
# Входы: нет явных аргументов.
# Результат: значение типа `FeatureConfig`.
def final_stable_feature_config() -> FeatureConfig:
    """Конфигурация основных признаков F0–F4."""
    return FeatureConfig(
        include_f0_raw=True,
        include_f1_categories=True,
        include_f2_time=True,
        include_snapshot_month=False,
        include_f3_missingness=True,
        include_f4_financial=True,
        excluded_raw_columns=(),
    )


# ФУНКЦИЯ `final_snapshot_feature_config`
# Создаёт stable-конфигурацию и включает единственный snapshot_month_index. Это временное представление
# третьего компонента V4.
# Входы: нет явных аргументов.
# Результат: значение типа `FeatureConfig`.
def final_snapshot_feature_config() -> FeatureConfig:
    """Основные признаки плюс индекс месяца снимка."""
    return replace(final_stable_feature_config(), include_snapshot_month=True)


# ФУНКЦИЯ `final_tuned_model_config`
# Возвращает зафиксированные гиперпараметры сильной tuned_90 L1-модели: 90 листьев, малый learning rate,
# sampling и регуляризацию.
# Входы: `num_threads`.
# Результат: значение типа `LightGBMConfig`.
def final_tuned_model_config(num_threads: int = 4) -> LightGBMConfig:
    """Основная конфигурация LightGBM для stable и snapshot моделей."""
    return LightGBMConfig(
        num_threads=num_threads,
        seed=SEED,
        learning_rate=0.016424199098700288,
        num_leaves=90,
        min_data_in_leaf=17,
        feature_fraction=0.6260206371941118,
        bagging_fraction=0.7096834432905521,
        bagging_freq=1,
        lambda_l1=3.4671276804481113,
        lambda_l2=4.905556676028774,
        max_bin=255,
        extra_params={
            "max_depth": 9,
            "min_sum_hessian_in_leaf": 0.26926469100861794,
            "min_gain_to_split": 1.6167946962329223,
        },
    )


# ФУНКЦИЯ `final_wide_model_config`
# Возвращает более широкую diversity-конфигурацию wide_95 с иным темпом обучения, sampling и регуляризацией.
# Входы: `num_threads`.
# Результат: значение типа `LightGBMConfig`.
def final_wide_model_config(num_threads: int = 4) -> LightGBMConfig:
    """Более широкая конфигурация для разнообразия ансамбля."""
    return LightGBMConfig(
        num_threads=num_threads,
        seed=SEED,
        learning_rate=0.03,
        num_leaves=95,
        min_data_in_leaf=24,
        feature_fraction=0.72,
        bagging_fraction=0.82,
        bagging_freq=1,
        lambda_l1=1.5,
        lambda_l2=6.0,
        max_bin=255,
        extra_params={"max_depth": 10, "min_gain_to_split": 0.15},
    )


# ФУНКЦИЯ `final_snapshot_model_config`
# Возвращает tuned-конфигурацию для snapshot-представления. Различие компонента задаётся признаками и числом
# раундов.
# Входы: `num_threads`.
# Результат: значение типа `LightGBMConfig`.
def final_snapshot_model_config(num_threads: int = 4) -> LightGBMConfig:
    """Параметры tuned-модели для snapshot-представления."""
    return final_tuned_model_config(num_threads)


# ФУНКЦИЯ `component_contract`
# Описывает три компонента V4, их раунды, веса и feature views и проверяет, что веса дают ровно единицу.
# Входы: нет явных аргументов.
# Результат: значение типа `dict[str, dict[str, Any]]`.
def component_contract() -> dict[str, dict[str, Any]]:
    """Возвращает число итераций, веса и представления компонентов."""
    components = {
        "tuned_90": {
            "rounds": FROZEN_ROUNDS_TUNED,
            "weight": TUNED_WEIGHT,
            "feature_view": "stable",
        },
        "wide_95": {
            "rounds": FROZEN_ROUNDS_WIDE,
            "weight": WIDE_WEIGHT,
            "feature_view": "stable",
        },
        "snapshot_tuned_90": {
            "rounds": FROZEN_ROUNDS_SNAPSHOT,
            "weight": SNAPSHOT_WEIGHT,
            "feature_view": "snapshot",
        },
    }
    if not abs(sum((spec["weight"] for spec in components.values())) - 1.0) <= 1e-12:
        raise RuntimeError("frozen component weights do not sum to one")
    return components


# ФУНКЦИЯ `final_component_specs`
# Объединяет сериализуемый component contract с объектами LightGBMConfig и возвращает runtime-spec для replay
# и final fit.
# Входы: `num_threads`.
# Результат: значение типа `dict[str, dict[str, Any]]`.
def final_component_specs(num_threads: int = 4) -> dict[str, dict[str, Any]]:
    """Добавляет к описанию компонентов конфигурации моделей."""
    specs = component_contract()
    configs = {
        "tuned_90": final_tuned_model_config(num_threads),
        "wide_95": final_wide_model_config(num_threads),
        "snapshot_tuned_90": final_snapshot_model_config(num_threads),
    }
    return {name: {**spec, "config": configs[name]} for name, spec in specs.items()}


In [9]:
# МОДУЛЬ 7. V5: residual LightGBM и source expert.
# Формирует признаки ошибки anchor, ограниченную ML-поправку и прозрачные денежные правила.

RESIDUAL_ROUNDS = 500
RESIDUAL_SHRINK = 1.0
MIN_HISTORY_ROWS = 5000
PREDICTION_LOWER = 20000.0
PREDICTION_UPPER = 1500000.0
PROXY_TOKENS = (
    "salary",
    "income",
    "payout",
    "turn",
    "balance",
    "amt",
    "amount",
    "sum",
    "median",
    "avg",
    "limit",
    "loan",
    "bki",
    "payment",
    "transaction",
    "profit",
    "cash",
    "credit",
    "debit",
    "rur",
)
GAP_TOKENS = (
    "salary",
    "income",
    "payout",
    "profit",
    "turn",
    "balance",
    "amt",
    "amount",
    "sum",
    "median",
    "avg",
)
MISSING_GROUPS = {
    "income": ("salary", "income", "payout", "profit"),
    "funds": ("balance", "turn", "amt", "amount", "sum", "rur"),
    "credit": ("loan", "credit", "bki", "overdue", "limit", "debt"),
    "transaction": ("transaction", "cashflow", "category", "purchase", "payment"),
    "employment": ("ils", "employ", "job", "seniority", "work"),
    "digital": ("vert_", "mob", "voice", "sms", "app", "ctn"),
}
SOURCE_SPEC = {
    "salary_scale": 0.95,
    "salary_low_ceiling": 75000.0,
    "salary_high_ceiling": 300000.0,
    "salary_mid_anchor_weight": 0.25,
    "income_scale": 1.2562141377310019,
    "payout_scale": 0.7831623092906007,
    "agreement_ratio": 1.4,
    "agreement_ceiling": 300000.0,
    "agreement_anchor_weight": 0.5,
}


# ФУНКЦИЯ `month_code_v5`
# Унифицирует разные представления месяца в year·12+month. Поддерживает Period, Timestamp, YYYYMM, внутренний
# integer и строку.
# Входы: `value`.
# Результат: значение типа `int`.
def month_code_v5(value) -> int:
    if isinstance(value, pd.Period):
        return value.year * 12 + value.month
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        stamp = pd.Timestamp(value)
        return stamp.year * 12 + stamp.month
    if isinstance(value, (int, np.integer)):
        number = int(value)
        if 190001 <= number <= 299912:
            return number // 100 * 12 + number % 100
        if 20000 <= number < 100000:
            return number
    digits = "".join((ch for ch in str(value) if ch.isdigit()))
    if len(digits) >= 6:
        return int(digits[:4]) * 12 + int(digits[4:6])
    period = pd.Period(str(value), freq="M")
    return period.year * 12 + period.month


# ФУНКЦИЯ `numeric_array_v5`
# Возвращает столбец source expert как float64; при отсутствии поля создаёт вектор NaN той же длины, сохраняя
# единый код правил.
# Входы: `frame`, `column`.
# Результат: значение типа `np.ndarray`.
def numeric_array_v5(frame: pd.DataFrame, column: str) -> np.ndarray:
    if column not in frame:
        return np.full(len(frame), np.nan, dtype=np.float64)
    return pd.to_numeric(frame[column], errors="coerce").to_numpy(np.float64)


# ФУНКЦИЯ `apply_source_expert_v5`
# Применяет прозрачные salary и income/payout правила к anchor, формирует gate-флаги и ограничивает ответ
# допустимым диапазоном. Возвращает prediction и причины срабатывания policy.
# Входы: `frame`, `anchor`.
# Результат: значение типа `tuple[np.ndarray, dict[str, np.ndarray]]`.
def apply_source_expert_v5(
    frame: pd.DataFrame, anchor
) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    base = np.asarray(anchor, dtype=np.float64)
    result = base.copy()
    salary = numeric_array_v5(frame, "salary_6to12m_avg")
    salary_proxy = SOURCE_SPEC["salary_scale"] * salary
    salary_low = (
        np.isfinite(salary_proxy)
        & (salary_proxy > 0)
        & (salary_proxy <= SOURCE_SPEC["salary_low_ceiling"])
    )
    salary_mid = (
        np.isfinite(salary_proxy)
        & (salary_proxy > SOURCE_SPEC["salary_low_ceiling"])
        & (salary_proxy <= SOURCE_SPEC["salary_high_ceiling"])
    )
    salary_gate = salary_low | salary_mid
    result[salary_low] = salary_proxy[salary_low]
    result[salary_mid] = (
        SOURCE_SPEC["salary_mid_anchor_weight"] * base[salary_mid]
        + (1.0 - SOURCE_SPEC["salary_mid_anchor_weight"]) * salary_proxy[salary_mid]
    )
    income = numeric_array_v5(frame, "incomeValue")
    payout = numeric_array_v5(frame, "dp_payoutincomedata_payout_avg_3_month")
    income_proxy = SOURCE_SPEC["income_scale"] * income
    payout_proxy = SOURCE_SPEC["payout_scale"] * payout
    positive_pair = (
        ~salary_gate
        & np.isfinite(income_proxy)
        & np.isfinite(payout_proxy)
        & (income_proxy > 0)
        & (payout_proxy > 0)
    )
    agreement = np.zeros(len(frame), dtype=bool)
    if positive_pair.any():
        log_ratio = np.zeros(len(frame), dtype=np.float64)
        log_ratio[positive_pair] = np.abs(
            np.log(income_proxy[positive_pair] / payout_proxy[positive_pair])
        )
        pair_anchor = np.full(len(frame), np.nan, dtype=np.float64)
        pair_anchor[positive_pair] = np.sqrt(
            income_proxy[positive_pair] * payout_proxy[positive_pair]
        )
        agreement = (
            positive_pair
            & (log_ratio <= np.log(SOURCE_SPEC["agreement_ratio"]))
            & (pair_anchor <= SOURCE_SPEC["agreement_ceiling"])
        )
        result[agreement] = (
            SOURCE_SPEC["agreement_anchor_weight"] * result[agreement]
            + (1.0 - SOURCE_SPEC["agreement_anchor_weight"]) * pair_anchor[agreement]
        )
    result = np.clip(result, PREDICTION_LOWER, PREDICTION_UPPER)
    return (
        result,
        {
            "salary_low": salary_low,
            "salary_mid": salary_mid,
            "salary_gate": salary_gate,
            "agreement_gate": agreement,
            "any_gate": salary_gate | agreement,
        },
    )


# ФУНКЦИЯ `select_numeric_columns_v5`
# Выбирает общие числовые поля train/test для residual recipe, исключая target, w, id и недоступный
# first_salary_income.
# Входы: `train_frame`, `test_frame`.
# Результат: значение типа `list[str]`.
def select_numeric_columns_v5(
    train_frame: pd.DataFrame, test_frame: pd.DataFrame
) -> list[str]:
    forbidden = {"target", "w", "id", "first_salary_income"}
    columns = [
        column
        for column in train_frame.columns
        if column not in forbidden
        and column in test_frame
        and is_numeric_dtype(train_frame[column])
        and is_numeric_dtype(test_frame[column])
        and test_frame[column].notna().any()
    ]
    if forbidden.intersection(columns):
        raise RuntimeError("forbidden feature entered residual schema")
    return columns


# ФУНКЦИЯ `signed_log_v5`
# Сжимает длинные числовые хвосты как sign(x)·log1p(|x|), сохраняя знак отрицательных значений.
# Входы: `values`.
# Результат: значение типа `np.ndarray`.
def signed_log_v5(values) -> np.ndarray:
    array = np.asarray(values, dtype=np.float64)
    return np.sign(array) * np.log1p(np.abs(array))


# ФУНКЦИЯ `fit_recipe_v5`
# На fit-части без target выбирает 212 raw numeric, 64 информативных missing flags, 80 signed-log и до 48
# anchor gaps; фиксирует итоговую размерность и median anchor.
# Входы: `fit_frame`, `test_reference`, `fit_anchor`.
# Результат: значение типа `dict`.
def fit_recipe_v5(
    fit_frame: pd.DataFrame, test_reference: pd.DataFrame, fit_anchor
) -> dict:
    numeric_columns = select_numeric_columns_v5(fit_frame, test_reference)
    numeric = fit_frame[numeric_columns].apply(pd.to_numeric, errors="coerce")
    numeric = numeric.replace([np.inf, -np.inf], np.nan)
    missing_rate = numeric.isna().mean()
    entropy = missing_rate * (1.0 - missing_rate)
    eligible_missing = entropy[(missing_rate >= 0.01) & (missing_rate <= 0.99)].sort_values(
        ascending=False
    )
    missing_columns = eligible_missing.index[:64].tolist()
    dynamic_records = []
    for column in numeric_columns:
        lowered = column.lower()
        if not any((token in lowered for token in PROXY_TOKENS)):
            continue
        values = numeric[column].to_numpy(np.float64)
        values = values[np.isfinite(values)]
        if len(values) < 500:
            continue
        q50, q99 = np.percentile(np.abs(values), [50, 99])
        ratio = (q99 + 1.0) / (q50 + 1.0)
        if q99 > 10.0 and ratio > 8.0:
            dynamic_records.append((float(ratio), column))
    dynamic_records.sort(reverse=True)
    signed_log_columns = [column for _, column in dynamic_records[:80]]
    gap_columns = [
        column
        for column in signed_log_columns
        if any((token in column.lower() for token in GAP_TOKENS))
    ][:48]
    safe_anchor = np.clip(np.asarray(fit_anchor, dtype=np.float64), 1.0, PREDICTION_UPPER)
    recipe = {
        "numeric_columns": numeric_columns,
        "missing_columns": missing_columns,
        "signed_log_columns": signed_log_columns,
        "gap_columns": gap_columns,
        "base_log_median": float(np.median(np.log1p(safe_anchor))),
        "feature_count": len(numeric_columns)
        + 2
        + len(MISSING_GROUPS)
        + len(missing_columns)
        + len(signed_log_columns)
        + 6
        + len(gap_columns),
    }
    del numeric
    return recipe


# ФУНКЦИЯ `transform_recipe_v5`
# По замороженному recipe строит residual-матрицу из raw, missingness, signed-log, anchor и gap блоков и
# проверяет уникальность/точное число столбцов.
# Входы: `frame`, `anchor`, `recipe`.
# Результат: значение типа `pd.DataFrame`.
def transform_recipe_v5(frame: pd.DataFrame, anchor, recipe: dict) -> pd.DataFrame:
    numeric_columns = recipe["numeric_columns"]
    numeric = frame[numeric_columns].apply(pd.to_numeric, errors="coerce")
    numeric = numeric.replace([np.inf, -np.inf], np.nan).astype(np.float32)
    parts = [numeric.reset_index(drop=True)]
    missing = numeric.isna()
    row_part = pd.DataFrame(
        {
            "__missing_count": missing.sum(axis=1).to_numpy(np.float32),
            "__missing_fraction": missing.mean(axis=1).to_numpy(np.float32),
        }
    )
    parts.append(row_part)
    group_part = {}
    for group_name, tokens in MISSING_GROUPS.items():
        group_columns = [
            column
            for column in numeric_columns
            if any((token in column.lower() for token in tokens))
        ]
        group_part[f"__missing_{group_name}"] = (
            missing[group_columns].sum(axis=1).to_numpy(np.float32)
        )
    parts.append(pd.DataFrame(group_part))
    flag_part = pd.DataFrame(
        {
            f"__isna__{column}": missing[column].to_numpy(np.float32)
            for column in recipe["missing_columns"]
        }
    )
    parts.append(flag_part)
    slog_part = pd.DataFrame(
        {
            f"__slog__{column}": signed_log_v5(numeric[column]).astype(np.float32)
            for column in recipe["signed_log_columns"]
        }
    )
    parts.append(slog_part)
    base = np.asarray(anchor, dtype=np.float64)
    safe_base = np.clip(base, 1.0, PREDICTION_UPPER)
    base_log = np.log1p(safe_base)
    rank = pd.Series(base).rank(method="average", pct=True).to_numpy(np.float64)
    rank_clipped = np.clip(rank, 0.0001, 1.0 - 0.0001)
    base_part = pd.DataFrame(
        {
            "__base_prediction": base.astype(np.float32),
            "__base_log1p": base_log.astype(np.float32),
            "__base_sqrt": np.sqrt(safe_base).astype(np.float32),
            "__base_rank": rank.astype(np.float32),
            "__base_rank_logit": np.log(
                rank_clipped / np.clip(1.0 - rank_clipped, 0.0001, 1.0)
            ).astype(np.float32),
            "__base_log_centered": (base_log - recipe["base_log_median"]).astype(
                np.float32
            ),
        }
    )
    parts.append(base_part)
    gap_part = pd.DataFrame(
        {
            f"__anchor_gap__{column}": (signed_log_v5(numeric[column]) - base_log).astype(
                np.float32
            )
            for column in recipe["gap_columns"]
        }
    )
    parts.append(gap_part)
    result = pd.concat(parts, axis=1)
    if result.columns.duplicated().any():
        raise RuntimeError("residual feature names are not unique")
    if result.shape[1] != recipe["feature_count"]:
        raise RuntimeError(
            f"Ожидалось {recipe['feature_count']} meta-признаков, получено {result.shape[1]}"
        )
    return result


# ФУНКЦИЯ `residual_params_v5`
# Возвращает frozen-параметры небольшой L1 LightGBM residual-модели V5: 15 листьев, глубина 4, 500 раундов и
# сильная регуляризация.
# Входы: нет явных аргументов.
# Результат: значение типа `dict`.
def residual_params_v5() -> dict:
    return {
        "objective": "regression_l1",
        "metric": "None",
        "verbosity": -1,
        "learning_rate": 0.025,
        "num_leaves": 15,
        "max_depth": 4,
        "min_data_in_leaf": 250,
        "min_sum_hessian_in_leaf": 25.0,
        "feature_fraction": 0.6,
        "feature_fraction_seed": 20260724,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "bagging_seed": 20260736,
        "lambda_l1": 20.0,
        "lambda_l2": 80.0,
        "min_gain_to_split": 2.0,
        "max_bin": 63,
        "data_random_seed": 20260750,
        "seed": 20260713,
        "num_threads": NUM_THREADS,
        "deterministic": True,
        "force_col_wise": True,
    }


# ФУНКЦИЯ `fit_residual_v5`
# Обучает LightGBM предсказывать residual=target−V4 anchor с исходными весами. Возвращает booster для
# исторического или финального correction.
# Входы: `X`, `residual`, `weights`.
# Результат: значение типа `lgb.Booster`.
def fit_residual_v5(X: pd.DataFrame, residual, weights) -> lgb.Booster:
    dataset = lgb.Dataset(
        X,
        label=np.asarray(residual, dtype=np.float64),
        weight=np.asarray(weights, dtype=np.float64),
        free_raw_data=False,
    )
    model = lgb.train(residual_params_v5(), dataset, num_boost_round=RESIDUAL_ROUNDS)
    del dataset
    return model


# ФУНКЦИЯ `correction_bounds_v5`
# Вычисляет q01/q99 residual на train и дополнительно ограничивает их безопасным диапазоном. Эти границы
# отсекают экстремальные поправки.
# Входы: `residual`.
# Результат: значение типа `tuple[float, float]`.
def correction_bounds_v5(residual) -> tuple[float, float]:
    lower, upper = np.quantile(np.asarray(residual, dtype=np.float64), [0.01, 0.99])
    return (max(float(lower), -250000.0), min(float(upper), 500000.0))


# ФУНКЦИЯ `score_v5`
# Считает official WMAE выбранного prediction-столбца внутри V5 replay.
# Входы: `frame`, `column`.
# Результат: значение типа `float`.
def score_v5(frame: pd.DataFrame, column: str) -> float:
    return wmae_official(frame.target, frame[column], frame.w)


In [10]:
# МОДУЛЬ 8. V6 Horizon Guard.
# Определяет обучение guard-коррекции и правила допуска поправки для горизонтов h1-h5.

GUARD_ROUNDS = 400
GUARD_RHO = 1.25
GUARD_LOWER = 20000.0
GUARD_UPPER = 1500000.0


# ФУНКЦИЯ `guard_wmae`
# Короткая DataFrame-обёртка official WMAE для V6 validation summaries.
# Входы: `frame`, `prediction`.
# Результат: значение типа `float`.
def guard_wmae(frame: pd.DataFrame, prediction) -> float:
    return float(wmae_official(frame.target, prediction, frame.w))


# ФУНКЦИЯ `transform_guard_v6`
# Строит 418 guard-признаков в той же координатной системе recipe, но сохраняет frozen dtype-путь V6.
# Отдельная реализация оставлена намеренно ради неизменности LightGBM результата.
# Входы: `frame`, `anchor`, `recipe`.
# Результат: значение типа `pd.DataFrame`.
def transform_guard_v6(frame: pd.DataFrame, anchor, recipe: dict) -> pd.DataFrame:
    columns = recipe["numeric_columns"]
    numeric = frame[columns].replace([np.inf, -np.inf], np.nan).astype(np.float32)
    missing = numeric.isna()
    parts = [numeric.reset_index(drop=True)]
    parts.append(
        pd.DataFrame(
            {
                "__missing_count": missing.sum(axis=1).to_numpy(np.float32),
                "__missing_fraction": missing.mean(axis=1).to_numpy(np.float32),
            }
        )
    )
    group_part = {}
    for name, tokens in MISSING_GROUPS.items():
        group_columns = [
            column
            for column in columns
            if any((token in column.lower() for token in tokens))
        ]
        group_part[f"__missing_{name}"] = (
            missing[group_columns].sum(axis=1).to_numpy(np.float32)
        )
    parts.append(pd.DataFrame(group_part))
    parts.append(
        pd.DataFrame(
            {
                f"__isna__{column}": missing[column].to_numpy(np.float32)
                for column in recipe["missing_columns"]
            }
        )
    )
    parts.append(
        pd.DataFrame(
            {
                f"__slog__{column}": signed_log_v5(numeric[column])
                for column in recipe["signed_log_columns"]
            }
        )
    )
    base = np.asarray(anchor, dtype=np.float64)
    safe = np.clip(base, 1.0, GUARD_UPPER)
    base_log = np.log1p(safe)
    rank = pd.Series(base).rank(method="average", pct=True).to_numpy(np.float64)
    rank_clip = np.clip(rank, 0.0001, 1.0 - 0.0001)
    parts.append(
        pd.DataFrame(
            {
                "__base_prediction": base.astype(np.float32),
                "__base_log1p": base_log.astype(np.float32),
                "__base_sqrt": np.sqrt(safe).astype(np.float32),
                "__base_rank": rank.astype(np.float32),
                "__base_rank_logit": np.log(rank_clip / (1.0 - rank_clip)).astype(
                    np.float32
                ),
                "__base_log_centered": (base_log - recipe["base_log_median"]).astype(
                    np.float32
                ),
            }
        )
    )
    parts.append(
        pd.DataFrame(
            {
                f"__anchor_gap__{column}": (
                    signed_log_v5(numeric[column]) - base_log
                ).astype(np.float32)
                for column in recipe["gap_columns"]
            }
        )
    )
    result = pd.concat(parts, axis=1)
    if result.columns.duplicated().any():
        raise RuntimeError("guard feature names are not unique")
    if result.shape[1] != recipe["feature_count"]:
        raise RuntimeError(
            f"Ожидалось {recipe['feature_count']} guard-признаков, получено {result.shape[1]}"
        )
    return result


# ФУНКЦИЯ `guard_params_v6`
# Возвращает frozen-параметры L1 LightGBM Horizon Guard с отдельным числом раундов и фиксированными seeds.
# Входы: нет явных аргументов.
# Результат: значение типа `dict`.
def guard_params_v6() -> dict:
    return {
        "objective": "regression_l1",
        "metric": "None",
        "verbosity": -1,
        "learning_rate": 0.025,
        "num_leaves": 15,
        "max_depth": 4,
        "min_data_in_leaf": 250,
        "min_sum_hessian_in_leaf": 25.0,
        "feature_fraction": 0.6,
        "feature_fraction_seed": 20260724,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "bagging_seed": 20260736,
        "lambda_l1": 20.0,
        "lambda_l2": 80.0,
        "min_gain_to_split": 2.0,
        "max_bin": 63,
        "data_random_seed": 20260750,
        "seed": 20260713,
        "num_threads": NUM_THREADS,
        "deterministic": True,
        "force_col_wise": True,
    }


# ФУНКЦИЯ `fit_guard_v6`
# Отбирает уникальные uncovered h1 OOT-наблюдения, строит target−source residual, обучает guard, вычисляет
# q01/q99 bounds и возвращает модель с metadata.
# Входы: `observations`.
# Результат: вычисленное значение, описанное в назначении функции.
def fit_guard_v6(observations: pd.DataFrame):
    history = observations.loc[
        observations.horizon.eq(1) & ~observations.source_any_gate.astype(bool)
    ].copy()
    history = history.sort_values(["target_month_code", "id"])
    if history.id.astype(str).duplicated().any():
        raise RuntimeError("guard h1 fit IDs are not globally unique")
    rows = history.row_index.to_numpy(np.int64)
    anchor = history.source_prediction.to_numpy(np.float64)
    X = transform_guard_v6(raw.train_features.iloc[rows], anchor, final_recipe_v5)
    residual = history.target.to_numpy(np.float64) - anchor
    lower, upper = np.quantile(residual, [0.01, 0.99])
    lower = max(float(lower), -250000.0)
    upper = min(float(upper), 500000.0)
    dataset = lgb.Dataset(
        X, label=residual, weight=history.w.to_numpy(np.float64), free_raw_data=False
    )
    model = lgb.train(guard_params_v6(), dataset, num_boost_round=GUARD_ROUNDS)
    meta = {
        "fit_rows": len(history),
        "unique_fit_ids": history.id.astype(str).nunique(),
        "target_month_min": int(history.target_month_code.min()),
        "target_month_max": int(history.target_month_code.max()),
        "correction_lower_q01": lower,
        "correction_upper_q99": upper,
    }
    del X, dataset, residual
    gc.collect()
    return (model, lower, upper, meta)


# ФУНКЦИЯ `predict_guard_v6`
# Строит guard-матрицу, предсказывает и ограничивает correction, обнуляет её на source-gated строках и
# применяет rho=1.25 к anchor.
# Входы: `model`, `frame`, `anchor`, `source_gate`, `lower`, `upper`.
# Результат: вычисленное значение, описанное в назначении функции.
def predict_guard_v6(
    model: lgb.Booster, frame: pd.DataFrame, anchor, source_gate, lower: float, upper: float
):
    X = transform_guard_v6(frame, anchor, final_recipe_v5)
    raw_correction = np.clip(model.predict(X), lower, upper)
    gate = np.asarray(source_gate, dtype=bool)
    applied_correction = np.where(gate, 0.0, raw_correction)
    prediction = np.clip(
        np.asarray(anchor, dtype=np.float64) + GUARD_RHO * applied_correction,
        GUARD_LOWER,
        GUARD_UPPER,
    )
    del X
    gc.collect()
    return prediction


# ФУНКЦИЯ `guard_score_summary`
# Считает source, V5, clean и guarded WMAE целиком и по каждому горизонту, включая относительный gain против
# V5.
# Входы: `frame`, `clean`, `guarded`.
# Результат: значение типа `dict`.
def guard_score_summary(frame: pd.DataFrame, clean, guarded) -> dict:
    source = frame.source_prediction.to_numpy(np.float64)
    v5 = frame.candidate_prediction.to_numpy(np.float64)
    source_score = guard_wmae(frame, source)
    v5_score = guard_wmae(frame, v5)
    clean_score = guard_wmae(frame, clean)
    guarded_score = guard_wmae(frame, guarded)
    cells = []
    scored = frame.assign(clean_prediction=clean, guarded_prediction=guarded)
    for horizon_number, cell in scored.groupby("horizon", sort=True):
        cell_v5 = guard_wmae(cell, cell.candidate_prediction)
        cell_clean = guard_wmae(cell, cell.clean_prediction)
        cell_guarded = guard_wmae(cell, cell.guarded_prediction)
        cells.append(
            {
                "horizon": int(horizon_number),
                "rows": len(cell),
                "source_wmae": guard_wmae(cell, cell.source_prediction),
                "v5_wmae": cell_v5,
                "clean_wmae": cell_clean,
                "guarded_wmae": cell_guarded,
                "clean_gain_vs_v5": (cell_v5 - cell_clean) / cell_v5,
                "guarded_gain_vs_v5": (cell_v5 - cell_guarded) / cell_v5,
            }
        )
    return {
        "rows": len(frame),
        "source_wmae": source_score,
        "v5_wmae": v5_score,
        "clean_wmae": clean_score,
        "guarded_wmae": guarded_score,
        "clean_gain_vs_v5": (v5_score - clean_score) / v5_score,
        "guarded_gain_vs_v5": (v5_score - guarded_score) / v5_score,
        "cells": cells,
    }


# ФУНКЦИЯ `replay_guard_v6`
# Обучает guard на более ранних observations, применяет к validation month, сохраняет exact V5 на h1 и exact
# source на gated h2–h5, затем возвращает summary и fit metadata.
# Входы: `train_obs`, `valid_obs`.
# Результат: вычисленное значение, описанное в назначении функции.
def replay_guard_v6(train_obs: pd.DataFrame, valid_obs: pd.DataFrame):
    model, lower, upper, fit_meta = fit_guard_v6(train_obs)
    rows = valid_obs.row_index.to_numpy(np.int64)
    clean = predict_guard_v6(
        model,
        raw.train_features.iloc[rows],
        valid_obs.source_prediction.to_numpy(np.float64),
        valid_obs.source_any_gate.to_numpy(bool),
        lower,
        upper,
    )
    v5 = valid_obs.candidate_prediction.to_numpy(np.float64)
    horizon = valid_obs.horizon.to_numpy(np.int16)
    guarded = np.where(horizon == 1, v5, clean)
    if not np.array_equal(guarded[horizon == 1], v5[horizon == 1]):
        raise RuntimeError("historical h1 was not preserved exactly")
    gate = valid_obs.source_any_gate.to_numpy(bool)
    future_gate = gate & (horizon >= 2)
    source = valid_obs.source_prediction.to_numpy(np.float64)
    if not np.array_equal(guarded[future_gate], source[future_gate]):
        raise RuntimeError("historical gated h2-h5 rows changed from source")
    summary = guard_score_summary(valid_obs, clean, guarded)
    del model
    gc.collect()
    return (summary, fit_meta)


## 4. V4: временной replay

Для каждого cutoff модели обучаются только на доступных более ранних месяцах. Это создаёт out-of-time прогнозы для обучения следующих уровней.


In [11]:
# МОДУЛЬ 9. Функции временного replay V4.
# Готовит stable/snapshot матрицы и выполняет out-of-time обучение трёх компонентов по cutoff.

feature_config = final_snapshot_feature_config()
component_specs = final_component_specs(NUM_THREADS)
component_weights = {name: float(spec["weight"]) for name, spec in component_specs.items()}


# ФУНКЦИЯ `matrix_views`
# Одним fold-local transformer строит snapshot-матрицу и производную stable-матрицу без snapshot_month_index.
# Возвращает оба представления с одинаковыми строками.
# Входы: `transformer`, `features`, `dates`, `fit`.
# Результат: вычисленное значение, описанное в назначении функции.
def matrix_views(transformer, features, dates, *, fit: bool):
    snapshot = (
        transformer.fit_transform(features, dates, include_native_categories=True)
        if fit
        else transformer.transform(features, dates, include_native_categories=True)
    )
    if SNAPSHOT_FEATURE_NAME not in snapshot.columns:
        raise RuntimeError("Не найден признак месяца снимка")

    stable = snapshot.drop(columns=[SNAPSHOT_FEATURE_NAME])
    return {"stable": stable, "snapshot": snapshot}


# ФУНКЦИЯ `component_frame`
# Упаковывает OOT-прогноз компонента V4 вместе с id, cutoff, target month, horizon, target и весом для
# последующего merge и residual replay.
# Входы: `cutoff`, `valid_idx`, `prediction`.
# Результат: вычисленное значение, описанное в назначении функции.
def component_frame(cutoff, valid_idx, prediction):
    target_month = raw.train_month[valid_idx]
    return pd.DataFrame(
        {
            "id": raw.train_ids.iloc[valid_idx].to_numpy(),
            "cutoff": month_label(cutoff),
            "target_month": [month_label(value) for value in target_month],
            "horizon": target_month - cutoff,
            "target": raw.y[valid_idx],
            "w": raw.w[valid_idx],
            "prediction": prediction,
        }
    )


In [12]:
# МОДУЛЬ 10. Запуск expanding replay V4.
# Строит OOT-прогнозы по историческим месяцам и объединяет их в доказательную replay-таблицу.

months_v4 = tuple(sorted(int(value) for value in np.unique(raw.train_month)))
if len(months_v4) < 2:
    raise RuntimeError("Для временного replay нужны минимум два месяца train")

component_parts = {name: [] for name in component_specs}

# На каждом cutoff модель видит только доступное прошлое.
for cutoff in months_v4[:-1]:
    train_idx = np.flatnonzero(raw.train_month <= cutoff)
    valid_idx = np.flatnonzero(raw.train_month > cutoff)
    transformer = TargetFreeFeatureTransformer(feature_config)

    train_views = matrix_views(
        transformer,
        raw.train_features.iloc[train_idx],
        raw.train_dates.iloc[train_idx],
        fit=True,
    )
    valid_views = matrix_views(
        transformer,
        raw.train_features.iloc[valid_idx],
        raw.train_dates.iloc[valid_idx],
        fit=False,
    )

    for name, spec in component_specs.items():
        view = spec["feature_view"]
        model = fit_lightgbm_fixed(
            train_views[view],
            raw.y[train_idx],
            raw.w[train_idx],
            raw.train_month[train_idx],
            rounds=spec["rounds"],
            config=spec["config"],
        )
        prediction = predict_lightgbm_fixed(model, valid_views[view])
        component_parts[name].append(component_frame(cutoff, valid_idx, prediction))
        del model

    del transformer, train_views, valid_views
    gc.collect()
    print(
        f"cutoff={month_label(cutoff)} | train={len(train_idx):,} | valid={len(valid_idx):,}"
    )

component_tables = {
    name: pd.concat(parts, ignore_index=True) for name, parts in component_parts.items()
}
keys_v4 = ["id", "cutoff", "target_month", "horizon"]
renamed = {
    name: table.rename(
        columns={
            "target": f"target__{name}",
            "w": f"w__{name}",
            "prediction": f"prediction__{name}",
        }
    )
    for name, table in component_tables.items()
}

merged = renamed["tuned_90"]
for name in ("wide_95", "snapshot_tuned_90"):
    merged = merged.merge(renamed[name], on=keys_v4, how="inner", validate="one_to_one")
    if not np.array_equal(merged["target__tuned_90"], merged[f"target__{name}"]):
        raise RuntimeError(f"Не совпал target компонента {name}")
    if not np.array_equal(merged["w__tuned_90"], merged[f"w__{name}"]):
        raise RuntimeError(f"Не совпали веса компонента {name}")

tuned_oot = merged["prediction__tuned_90"].to_numpy(np.float64)
wide_oot = merged["prediction__wide_95"].to_numpy(np.float64)
snapshot_oot = merged["prediction__snapshot_tuned_90"].to_numpy(np.float64)

horizon_v4 = (
    pd.DataFrame(
        {
            "id": merged["id"],
            "cutoff": merged["cutoff"],
            "target_month": merged["target_month"],
            "horizon": merged["horizon"].astype(np.int8),
            "target": merged["target__tuned_90"],
            "w": merged["w__tuned_90"],
            "prediction": 0.36 * tuned_oot + 0.24 * wide_oot + 0.40 * snapshot_oot,
        }
    )
    .sort_values(keys_v4)
    .reset_index(drop=True)
)

del component_parts, component_tables, renamed, merged, tuned_oot, wide_oot, snapshot_oot
gc.collect()
print(f"V4 replay: {len(horizon_v4):,} строк")


cutoff=2024-01 | train=7,243 | valid=69,543
cutoff=2024-02 | train=16,108 | valid=60,678
cutoff=2024-03 | train=29,521 | valid=47,265
cutoff=2024-04 | train=44,379 | valid=32,407
cutoff=2024-05 | train=60,572 | valid=16,214
V4 replay: 226,107 строк


## 5. V4: обучение на всём train


In [13]:
# МОДУЛЬ 11. Финальное обучение V4 на всём train.
# Переобучает три компонента на полной истории и создаёт базовый test-anchor для следующих слоёв.

final_transformer = TargetFreeFeatureTransformer(feature_config)
full_views = matrix_views(final_transformer, raw.train_features, raw.train_dates, fit=True)
test_views = matrix_views(final_transformer, raw.test_features, raw.test_dates, fit=False)

# Финальные модели обучаются на всём train; сохранять и загружать их повторно не нужно.
component_predictions = {}
for name, spec in component_specs.items():
    view = spec["feature_view"]
    model = fit_lightgbm_fixed(
        full_views[view],
        raw.y,
        raw.w,
        raw.train_month,
        rounds=spec["rounds"],
        config=spec["config"],
    )
    component_predictions[name] = predict_lightgbm_fixed(model, test_views[view])
    del model

base_prediction_v4 = blend_component_predictions(component_predictions, component_weights)

del final_transformer, full_views, test_views, component_predictions
gc.collect()
print("Финальный V4 готов")


Финальный V4 готов


## 6. V5: source expert и residual correction

Residual-модель обучается на уникальных h1 out-of-time ошибках. Source expert использует отдельные зарплатные и доходные правила.


In [14]:
# МОДУЛЬ 12. Временной replay V5.
# Строит rolling residual history, source gates и локальные сравнения V4/source/V5 без утечки.

horizon_v5 = horizon_v4.copy()
horizon_v5["cutoff_code"] = horizon_v5["cutoff"].map(month_code_v5)
horizon_v5["target_month_code"] = horizon_v5["target_month"].map(month_code_v5)
horizon_v5["horizon"] = horizon_v5["horizon"].astype(int)
if horizon_v5.duplicated(["cutoff", "target_month", "id"]).any():
    raise RuntimeError("В V4 replay обнаружены дубликаты")

id_to_row_v5 = pd.Series(
    np.arange(len(raw.train_ids), dtype=np.int64),
    index=raw.train_ids.astype(str).to_numpy(),
)
horizon_ids_v5 = horizon_v5["id"].astype(str)
if not horizon_ids_v5.isin(id_to_row_v5.index).all():
    raise RuntimeError("В V4 replay обнаружены неизвестные id")

horizon_v5["row_index"] = id_to_row_v5.loc[horizon_ids_v5].to_numpy(np.int64)
h1_v5 = horizon_v5.loc[horizon_v5["horizon"].eq(1)].copy()
if h1_v5["id"].duplicated().any() or h1_v5.empty:
    raise RuntimeError("Для V5 нужны уникальные h1-наблюдения")

rolling_parts_v5 = []
for cutoff_code in sorted(horizon_v5["cutoff_code"].unique()):
    current = horizon_v5.loc[horizon_v5["cutoff_code"].eq(cutoff_code)].copy()
    current_index = current["row_index"].to_numpy(np.int64)
    current_features = raw.train_features.iloc[current_index]

    source_prediction, current_gates = apply_source_expert_v5(
        current_features, current["prediction"]
    )
    current["source_prediction"] = source_prediction
    current["candidate_prediction"] = source_prediction
    current["source_any_gate"] = current_gates["any_gate"]

    history = h1_v5.loc[h1_v5["target_month_code"].le(cutoff_code)].copy()
    if len(history) >= MIN_HISTORY_ROWS:
        history_index = history["row_index"].to_numpy(np.int64)
        history_features = raw.train_features.iloc[history_index]
        history_anchor = history["prediction"].to_numpy(np.float64)

        recipe = fit_recipe_v5(history_features, raw.test_features, history_anchor)
        X_history = transform_recipe_v5(history_features, history_anchor, recipe)
        X_current = transform_recipe_v5(
            current_features, current["prediction"].to_numpy(np.float64), recipe
        )
        residual = history["target"].to_numpy(np.float64) - history_anchor
        lower, upper = correction_bounds_v5(residual)
        model = fit_residual_v5(X_history, residual, history["w"])
        correction = np.clip(model.predict(X_current), lower, upper)
        corrected_base = np.clip(
            current["prediction"].to_numpy(np.float64) + RESIDUAL_SHRINK * correction,
            PREDICTION_LOWER,
            PREDICTION_UPPER,
        )
        candidate, _ = apply_source_expert_v5(current_features, corrected_base)

        current["candidate_prediction"] = candidate
        del history_features, X_history, X_current, recipe, model, residual
        gc.collect()

    rolling_parts_v5.append(current)

rolling_v5 = pd.concat(rolling_parts_v5, ignore_index=True)
if len(rolling_v5) != len(horizon_v5):
    raise RuntimeError("V5 replay потерял строки")

overall_v4_v5 = score_v5(rolling_v5, "prediction")
overall_source_v5 = score_v5(rolling_v5, "source_prediction")
overall_candidate_v5 = score_v5(rolling_v5, "candidate_prediction")

v5_summary = pd.DataFrame(
    [
        {
            "v4_wmae": overall_v4_v5,
            "source_wmae": overall_source_v5,
            "candidate_wmae": overall_candidate_v5,
            "source_gain_vs_v4": (overall_v4_v5 - overall_source_v5) / overall_v4_v5,
            "candidate_gain_vs_v4": (overall_v4_v5 - overall_candidate_v5) / overall_v4_v5,
            "candidate_gain_vs_source": (overall_source_v5 - overall_candidate_v5)
            / overall_source_v5,
        }
    ]
)
display(v5_summary)


,v4_wmae,source_wmae,candidate_wmae,source_gain_vs_v4,candidate_gain_vs_v4,candidate_gain_vs_source
0,64853.735261,63206.898571,62683.374431,0.025393,0.033465,0.008283


In [15]:
# МОДУЛЬ 13. Финальное обучение residual-модели V5.
# Обучает V5 на уникальных h1 OOT-ошибках и рассчитывает V5 candidate для test.

full_h1_v5 = h1_v5.sort_values("target_month_code").copy()
full_index_v5 = full_h1_v5["row_index"].to_numpy(np.int64)
full_features_v5 = raw.train_features.iloc[full_index_v5]
full_anchor_v5 = full_h1_v5["prediction"].to_numpy(np.float64)

final_recipe_v5 = fit_recipe_v5(full_features_v5, raw.test_features, full_anchor_v5)
X_meta_train_v5 = transform_recipe_v5(full_features_v5, full_anchor_v5, final_recipe_v5)
X_meta_test_v5 = transform_recipe_v5(raw.test_features, base_prediction_v4, final_recipe_v5)
full_residual_v5 = full_h1_v5["target"].to_numpy(np.float64) - full_anchor_v5
final_lower_v5, final_upper_v5 = correction_bounds_v5(full_residual_v5)

final_meta_model_v5 = fit_residual_v5(X_meta_train_v5, full_residual_v5, full_h1_v5["w"])
test_correction_v5 = np.clip(
    final_meta_model_v5.predict(X_meta_test_v5),
    final_lower_v5,
    final_upper_v5,
)
corrected_test_base_v5 = np.clip(
    base_prediction_v4 + RESIDUAL_SHRINK * test_correction_v5,
    PREDICTION_LOWER,
    PREDICTION_UPPER,
)

source_test_v5, _ = apply_source_expert_v5(raw.test_features, base_prediction_v4)
candidate_test_v5, source_test_gates_v5 = apply_source_expert_v5(
    raw.test_features, corrected_test_base_v5
)
test_month_code_v5 = np.array(
    [month_code_v5(value) for value in raw.test_month], dtype=np.int64
)

del X_meta_train_v5, X_meta_test_v5, final_meta_model_v5
gc.collect()
print("Финальный V5 готов")


Финальный V5 готов


## 7. V6: Horizon Guard

Для h1 сохраняется прогноз V5. Для h2 и дальше применяется дополнительная residual-коррекция; строки source gate остаются без неё.


In [16]:
# МОДУЛЬ 14. Selection и независимый audit V6.
# Проверяет horizon policy сначала на мае, затем без перенастройки подтверждает её на июне.

rolling_guard_v6 = rolling_v5.copy()
rolling_guard_v6["source_any_gate"] = rolling_guard_v6["source_any_gate"].astype(bool)
months_guard_v6 = sorted(
    int(value) for value in rolling_guard_v6["target_month_code"].unique()
)
if len(months_guard_v6) < 2:
    raise RuntimeError("Для проверки Horizon Guard нужны минимум два месяца")

selection_month_v6 = months_guard_v6[-2]
audit_month_v6 = months_guard_v6[-1]
selection_summary_v6, selection_fit_v6 = replay_guard_v6(
    rolling_guard_v6.loc[rolling_guard_v6["target_month_code"].lt(selection_month_v6)],
    rolling_guard_v6.loc[rolling_guard_v6["target_month_code"].eq(selection_month_v6)],
)
audit_summary_v6, audit_fit_v6 = replay_guard_v6(
    rolling_guard_v6.loc[rolling_guard_v6["target_month_code"].lt(audit_month_v6)],
    rolling_guard_v6.loc[rolling_guard_v6["target_month_code"].eq(audit_month_v6)],
)

validation_summary_v6 = pd.DataFrame(
    [
        {
            "split": "selection",
            "fit_rows": selection_fit_v6["fit_rows"],
            **{key: value for key, value in selection_summary_v6.items() if key != "cells"},
        },
        {
            "split": "audit",
            "fit_rows": audit_fit_v6["fit_rows"],
            **{key: value for key, value in audit_summary_v6.items() if key != "cells"},
        },
    ]
)
display(validation_summary_v6)


,split,fit_rows,rows,source_wmae,v5_wmae,clean_wmae,guarded_wmae,clean_gain_vs_v5,guarded_gain_vs_v5
0,selection,29225,64772,61466.945560,60804.630618,59784.721512,59687.050092,0.016774,0.018380
1,audit,41469,81070,62563.622837,62026.935459,60985.741674,60952.943848,0.016786,0.017315


In [17]:
# МОДУЛЬ 15. Финальный V6 и формирование submission.csv.
# Обучает guard на доступной OOT-истории, выбирает маршрут h1-h5 и сохраняет прогноз в исходном порядке id.

final_guard_model_v6, final_lower_v6, final_upper_v6, _ = fit_guard_v6(rolling_guard_v6)
clean_test_v6 = predict_guard_v6(
    final_guard_model_v6,
    raw.test_features,
    source_test_v5,
    source_test_gates_v5["any_gate"],
    final_lower_v6,
    final_upper_v6,
)

unique_test_months_v6 = sorted(int(value) for value in np.unique(test_month_code_v5))
month_to_horizon_v6 = {
    month: index + 1 for index, month in enumerate(unique_test_months_v6)
}
test_horizon_v6 = np.array(
    [month_to_horizon_v6[int(value)] for value in test_month_code_v5],
    dtype=np.int16,
)

# H1 остаётся V5, а дальние горизонты получают коррекцию Horizon Guard.
final_prediction = np.where(test_horizon_v6 == 1, candidate_test_v5, clean_test_v6)

submission = sample_submission.copy()
submission["predict"] = final_prediction
submission.to_csv(SUBMISSION_PATH, sep=";", decimal=",", index=False)

print(f"Готово: {SUBMISSION_PATH}")
display(submission.head())


Готово: /content/submission.csv


,id,predict
0,0,43818.378085
1,1,41084.172429
2,3,30368.803183
3,9,72397.119262
4,11,45436.738340


## Результат

Файл будет сохранён как `/content/submission.csv`. Его можно скачать через панель **Files** в Colab.
